# e1-evolve-pilot - Notebook 1 of the ALICE-Next-E plan (`model/05-risks-and-90-day-plan.md` s3)

**Purpose.** Resolve the three Tier-0 unknowns before any money is spent: (1) the *real* tokens/s and the Kaggle quota,
(2) whether operator behaviour (emit `CALC(..)` / `LOOKUP(<verbatim key>)` exactly when warranted, stay silent otherwise, say so when a tool is not declared)
moves into the weights, (3) whether periphery-only tuning is enough at 2B. It also proves the export chain (QAT checkpoint -> the layout
`aegis-forge/repack_ternary.py` accepts). **It is not a ship step.**

**Trains** `microsoft/bitnet-b1.58-2B-4T-bf16` (MIT; the bf16 master) with *online BitLinear QAT written in this notebook* (the exact STE of
`tinybit/model.py`: per-tensor `gamma = mean|w|`, `w_q = round(w/gamma).clamp(-1,1)`, per-token int8 absmax activations), on gateway-verified
episodes, in three arms: **P** periphery only (all ternary weights frozen; embeddings, norms, SubLN gains train), **L** full QAT at LR 3e-5, **H** full QAT at LR 1e-4.

**Two modes, same code.** On a Kaggle GPU the notebook runs the real thing. With the environment variable `E1_SMOKE=1` it runs on CPU with a *tiny randomly
initialised BitNet* (same architecture family, same code paths, no 4.8 GB download) and tiny sizes; smoke numbers are plumbing checks, **never measurements**.

| Cell | What |
|---|---|
| 0 | environment, quota reading, QAT primitives and the CIS activation-quantization audit (G5), fp16-vs-bf16 numerics check (G12), real tokens/s of the base forward |
| 1 | data factory: gateway-verified episodes, generator field, SHA-256 manifest |
| 2 | tokenizer parity G1: Python port of the engine's `encode()` over the pruned 50,256-id vocabulary |
| 3 | base reference + the three training arms with kill gates, checkpointing and resume |
| 4 | export for `repack_ternary.py`, exact Linux-side commands, SHA-256 of every file |
| 5 | gates vs measured table -> `results.json` |

**Memory honesty.** Arms **L** and **H** keep fp32 master weights (2.41 B parameters = 9.6 GB) plus fp32 gradients plus 8-bit Adam state: about 24 GB before activations.
That is the proven `2 x T4` recipe of the plan (`device_map=auto`); it does **not** fit one 16 GB card. On a single T4/P100 only arm **P** (and the baseline) can run;
the notebook checks the numbers before it starts an arm and skips (with the reason in `results.json`) rather than crashing 3 h in.

**Knobs** (environment variables; Kaggle has no UI for them, so put `import os; os.environ["E1_ARMS"] = "P"` in a first cell):

| variable | meaning |
|---|---|
| `E1_SMOKE=1` | CPU smoke mode: tiny random BitNet, tiny sizes, plumbing check only |
| `E1_ARMS=P,L,H` | which arms to run in this session (default all three; one arm per session is the safe split) |
| `E1_OUT` | output directory (default `/kaggle/working`, else `./e1_out`) |
| `E1_DATA_DIR` | a previously frozen dataset directory (manifest SHA-256s are verified, generation is skipped) |
| `E1_RESUME_FROM` | a previous run's output directory to resume checkpoints from (add it as a Kaggle input) |
| `E1_MAX_HOURS` / `E1_MAX_STEPS` | stop cleanly (checkpoint written) after this many session hours (default 11) / optimizer steps |
| `E1_QUOTA_BEFORE_H` | the GPU hours left as shown by the Kaggle session panel, if the quota API is not readable |
| `E1_KEEP_CKPT=1` | keep the fp32 training checkpoint after an arm finishes (default: deleted, the export is the artifact) |
| `E1_SEED`, `E1_BASE_REPO`, `E1_OFFLINE`, `E1_WIKITEXT`, `E1_REPACKER`, `E1_CHECK_PACKED` | seed, base repo id, skip the tokenizer download (smoke), force WikiText-2 in smoke, path of `repack_ternary.py`, set 0 to skip the packed-incumbent trit diagnostic |

## Cell 0 - environment, QAT primitives, numerics check (G12), real tokens/s

In [1]:
import os, sys, json, math, hashlib, random, re, time, struct, shutil, gc, copy, statistics, subprocess, platform, itertools, warnings
from collections import Counter, defaultdict

SMOKE = os.environ.get("E1_SMOKE") == "1"
if SMOKE:       # a smoke run must not compete with other jobs on a shared CPU box: single-threaded math (set before torch is imported)
    for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
        os.environ[_v] = "1"
# HF's online BitLinear wraps torch.compile; compiling is unnecessary here and costs minutes on first call. Must be set before torch is imported.
os.environ.setdefault("TORCHDYNAMO_DISABLE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*IProgress.*")


def _pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    print("pip", pkgs, "rc", r.returncode)
    return r.returncode == 0


T_SESSION = time.time()
from importlib.metadata import version as _ver
try:
    _tv = int(_ver("transformers").split(".")[0])
except Exception:
    _tv = 0
if _tv < 5:      # the plan's recipe: transformers >= 5.x native BitNet (`dtype=` kwarg, `trust_remote_code` off). Upgrade BEFORE the first import.
    assert not SMOKE, "smoke mode needs transformers >= 5 (pip install -U transformers)"
    _pip("-U", "transformers>=5")
import transformers
from transformers import AutoConfig, AutoModelForCausalLM
assert hasattr(transformers, "BitNetForCausalLM"), "this transformers build has no native BitNet"
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    transformers.utils.logging.disable_progress_bar()
    transformers.utils.logging.set_verbosity_error()
except Exception:
    pass
if SMOKE:
    torch.set_num_threads(1)
BASE_REPO = os.environ.get("E1_BASE_REPO", "microsoft/bitnet-b1.58-2B-4T-bf16")
WORK = os.environ.get("E1_OUT") or ("/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./e1_out"))
os.makedirs(WORK, exist_ok=True)
CUDA = torch.cuda.is_available() and not SMOKE
DEVICE0 = torch.device("cuda:0") if CUDA else torch.device("cpu")
N_GPU = torch.cuda.device_count() if CUDA else 0

CFG = dict(seq=1024, mb=1, accum=16, warmup=10, eval_every=100, ckpt_every=100, max_steps=None, target_tokens=14_000_000, n_episodes=None,
           dev_per_bucket=50, ppl_seqs=200, ppl_len=512, parity_seqs=50, gen_new=24, g1_vectors=20000, min_lr_frac=0.1, clip=1.0, wd=0.0,
           betas=(0.9, 0.95), gen_bs=32, ppl_bs=4, max_hours=float(os.environ.get("E1_MAX_HOURS", "11.0")),
           lr={"P": 1e-4, "L": 3e-5, "H": 1e-4})
if SMOKE:
    CFG.update(seq=128, mb=2, accum=2, warmup=2, eval_every=6, ckpt_every=6, max_steps=12, target_tokens=0, n_episodes=1200, dev_per_bucket=4,
               ppl_seqs=8, ppl_len=64, parity_seqs=4, g1_vectors=200, gen_bs=8, ppl_bs=4, lr={"P": 3e-3, "L": 1e-3, "H": 3e-3})
if os.environ.get("E1_MAX_STEPS"):
    CFG["max_steps"] = int(os.environ["E1_MAX_STEPS"])
ARMS = [a for a in os.environ.get("E1_ARMS", "P,L,H").split(",") if a]
print("mode:", "SMOKE (CPU, tiny random model; numbers are plumbing checks, not measurements)" if SMOKE else "REAL (Kaggle GPU)")

# ------------------------------------------------------------------ environment log
ENV = {"mode": "smoke" if SMOKE else "real", "python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
       "base_repo": BASE_REPO, "cuda": bool(CUDA), "n_gpu": N_GPU, "gpus": []}
if CUDA:
    for i in range(N_GPU):
        free, total = torch.cuda.mem_get_info(i)
        p = torch.cuda.get_device_properties(i)
        ENV["gpus"].append({"index": i, "name": p.name, "capability": f"{p.major}.{p.minor}", "total_gib": round(total / 2 ** 30, 2), "free_gib": round(free / 2 ** 30, 2)})
    ENV["bf16_native"] = bool(torch.cuda.is_bf16_supported(including_emulation=False))
    try:
        ENV["nvidia_smi"] = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version", "--format=csv,noheader"], capture_output=True, text=True, timeout=20).stdout.strip()
    except Exception:
        pass
else:
    ENV["bf16_native"] = True
AMP_DTYPE = torch.bfloat16 if ENV["bf16_native"] else torch.float16        # T4 / P100 have no native bf16 -> fp16 autocast + GradScaler
USE_SCALER = (AMP_DTYPE == torch.float16) and CUDA
ENV["amp_dtype"] = str(AMP_DTYPE)
try:
    import bitsandbytes as bnb
    ENV["bitsandbytes"] = bnb.__version__
except Exception:
    bnb = None
    if CUDA:
        _pip("bitsandbytes")
        try:
            import bitsandbytes as bnb
            ENV["bitsandbytes"] = bnb.__version__
        except Exception as e:  # noqa
            ENV["bitsandbytes"] = "unavailable: " + type(e).__name__
if CUDA and bnb is not None:       # pre-flight: does the 8-bit paged optimizer actually run on this GPU? (P100 / old bitsandbytes builds may not)
    try:
        _p = nn.Parameter(torch.randn(8192, device=DEVICE0))
        _o = bnb.optim.PagedAdamW8bit([_p], lr=1e-3)
        _p.grad = torch.randn_like(_p)
        _o.step()
        torch.cuda.synchronize()
        ENV["bnb_8bit_optimizer_ok"] = True
        del _p, _o
    except Exception as e:  # noqa
        ENV["bnb_8bit_optimizer_ok"] = f"unusable: {type(e).__name__}: {str(e)[:100]}"
        bnb = None
print(json.dumps(ENV, indent=1))


def read_quota():
    """Kaggle weekly-quota reading. Order: manual override (what the session panel shows), then the kaggle API with Kaggle secrets. The raw API
    'allowed' figure reads 6 h while kernels ran past it (cm-kaggle), so it is recorded raw and never trusted as the cap."""
    q = {"status": "unavailable"}
    if SMOKE:
        return {"status": "not applicable in smoke mode"}
    for k in ("E1_QUOTA_BEFORE_H", "E1_QUOTA_HOURS_LEFT"):
        if os.environ.get(k):
            q = {"status": "manual", "hours_left_as_shown_in_session_panel": float(os.environ[k]), "source_env": k}
    try:
        try:
            from kaggle_secrets import UserSecretsClient
            us = UserSecretsClient()
            for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                if not os.environ.get(k):
                    os.environ[k] = us.get_secret(k)
        except Exception:
            pass
        from kaggle.api.kaggle_api_extended import KaggleApi
        a = KaggleApi()
        a.authenticate()
        v = a.quota_view()
        q["api"] = {"gpu_quota": str(getattr(v, "gpu_quota", None)), "tpu_quota": str(getattr(v, "tpu_quota", None)),
                    "refresh": str(getattr(v, "quota_refresh_time", None))}
        q["status"] = "api" if q["status"] == "unavailable" else q["status"] + "+api"
    except Exception as e:  # noqa
        q["api_error"] = type(e).__name__ + ": " + str(e)[:120]
    return q


QUOTA_BEFORE = read_quota()
print("E1_QUOTA_BEFORE", json.dumps(QUOTA_BEFORE))

# ------------------------------------------------------------------ QAT primitives: verbatim math of tinybit/model.py
def weight_ternary_gamma(w):
    return w.abs().mean().clamp(min=1e-8)


def snap_ternary(w, gamma):
    return torch.round(w / gamma).clamp_(-1.0, 1.0)          # round-half-to-even, as numpy.rint in the repacker


def weight_fake_quant(w, keep_fp32=False):
    """per-tensor absmean ternary with STE: w_eff = w + (w_q*gamma - w).detach(); computed in fp32. Under autocast the fp32 result is returned so F.linear rounds it exactly once."""
    w32 = w.float()
    gamma = weight_ternary_gamma(w32)
    w_q = snap_ternary(w32, gamma)
    r = w32 + (w_q * gamma - w32).detach()
    return r if keep_fp32 else r.to(w.dtype)


def act_fake_quant(x):
    """per-token (last dim) absmax int8 quantize->dequantize with STE; s = 127/absmax, clamp [-127,127] (ops.rs quantize_activations_int8)."""
    x32 = x.float()
    absmax = x32.abs().amax(dim=-1, keepdim=True).clamp(min=1e-5)
    s = 127.0 / absmax
    x_q = torch.round(x32 * s).clamp_(-127.0, 127.0) / s
    return (x32 + (x_q - x32).detach()).to(x.dtype)


def grid_error(xq):
    """max distance of a quantized activation from the int8 grid, and the smallest per-token peak (must be 127)."""
    x32 = xq.float()
    s = 127.0 / x32.abs().amax(dim=-1, keepdim=True).clamp(min=1e-5)
    z = x32 * s
    return float((z - z.round()).abs().max()), float(z.abs().amax(dim=-1).min())


QAT_PROBE = None     # set to a dict to record G5 evidence during one forward


class QATLinear(nn.Module):
    """Bias-free BitLinear: activation fake-quant on the INPUT, ternary fake-quant on the weight. Shares the original Parameter (state_dict keys unchanged)."""

    def __init__(self, lin, name):
        super().__init__()
        assert lin.bias is None, "the engine has no bias path"
        self.weight = lin.weight
        self.in_features, self.out_features, self.qname = lin.in_features, lin.out_features, name
        self.exported = False          # True: weight already holds w_q * scale (loaded back from an export), do not fake-quant it again

    def forward(self, x):
        xq = act_fake_quant(x)
        if QAT_PROBE is not None:
            QAT_PROBE[self.qname] = grid_error(xq)
        w = self.weight if self.exported else weight_fake_quant(self.weight, keep_fp32=torch.is_autocast_enabled(xq.device.type))
        if w.dtype != xq.dtype and not torch.is_autocast_enabled(xq.device.type):
            w = w.to(xq.dtype)          # fp16 frozen body outside autocast (self-checks run in fp32); under autocast F.linear casts both operands itself
        return F.linear(xq, w)


PROJ = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


def wrap_qat(model):
    n = 0
    for name, mod in list(model.named_modules()):
        for pn in PROJ:
            child = getattr(mod, pn, None)
            if isinstance(child, nn.Linear):
                assert type(child) is nn.Linear, f"{name}.{pn} is a {type(child).__name__}: the checkpoint's quantization_config was not stripped before loading"
                setattr(mod, pn, QATLinear(child, f"{name}.{pn}"))
                n += 1
    assert n == 7 * model.config.num_hidden_layers, f"wrapped {n} projections, expected {7 * model.config.num_hidden_layers}"
    return n


def backbone(model):
    return model.model


def embed_weight(model):
    return model.get_input_embeddings().weight


def head_logits(model, h, kept_idx, head_quant=True):
    """Logits over the engine's served ids only (G6). The final-norm output is int8-quantized before the tied head (CIS point 5, G5)."""
    W = embed_weight(model)
    Wk = W.index_select(0, kept_idx.to(W.device))
    h = h.to(W.device)
    if head_quant:
        h = act_fake_quant(h)
        if QAT_PROBE is not None:
            QAT_PROBE["<head input: final-norm output>"] = grid_error(h)
    return F.linear(h, Wk)


def strip_quantization_config(cfg):
    if getattr(cfg, "quantization_config", None) is not None:
        cfg.quantization_config = None
    return cfg


# ------------------------------------------------------------------ the base checkpoint (real: the hub repo; smoke: a tiny random BitNet with the same layout)
def make_tiny_base(vocab, path):
    from transformers import BitNetConfig, BitNetForCausalLM
    torch.manual_seed(0)
    cfg = BitNetConfig(vocab_size=vocab, hidden_size=64, intermediate_size=128, num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
                       hidden_act="relu2", max_position_embeddings=512, rms_norm_eps=1e-5, tie_word_embeddings=True)
    m = BitNetForCausalLM(cfg)
    with torch.no_grad():
        for n, p in m.named_parameters():       # real BitNet masters are large (mean|w| ~ 1-2); mimic the scale so the ternary pattern is non-degenerate
            if p.dim() == 2 and "embed" not in n:
                p.mul_(40.0)
            elif "embed" in n:
                p.normal_(0.0, 0.3)             # spread the (tied) logits so argmax is not decided by numerical noise on a random net
            p.copy_(p.to(torch.bfloat16).float())   # a bf16 master, like the real checkpoint (so the fp16 container of arm P is lossless)
    m.config.quantization_config = {"quant_method": "bitnet", "linear_class": "autobitlinear", "quantization_mode": "online"}
    m.save_pretrained(path)
    return path


BASE_SRC = BASE_REPO
SMOKE_VOCAB = 128256
if SMOKE:
    BASE_SRC = make_tiny_base(SMOKE_VOCAB, os.path.join(WORK, "tiny_base"))
    print("[smoke] tiny random BitNet saved to", BASE_SRC)
KEPT_REAL = torch.tensor(list(range(50000)) + list(range(128000, 128256)), dtype=torch.long)

# ------------------------------------------------------------------ G12: fp16-vs-bf16 numerics on 20 prompts (run on the QAT forward that training uses)
G12_CANDIDATES = [
    "The quick brown fox jumps over the lazy dog.", "Explain why the sky is blue in two sentences.", "Q: What is the capital of France?\nA:",
    "Write a short note about washing your hands before eating.", "The meeting is scheduled for Tuesday at noon in the main office.",
    "Once upon a time there was a small town by the sea.", "Water boils at one hundred degrees Celsius at sea level.",
    "List three things you need to bake a loaf of bread.", "The engineer checked the pressure gauge twice before opening the valve.",
    "Q: How many days are in a week?\nA:", "She walked to the market and bought apples, bread and cheese.", "In 1969 humans first walked on the moon.",
    "A good operator reads the manual before starting the machine.", "Please summarize the following report in one paragraph.",
    "The train leaves at half past nine and arrives before lunch.", "Q: What do bees make?\nA:", "Always label the samples and keep the log up to date.",
    "Rain is expected tomorrow, so bring an umbrella.", "The library closes early on public holidays.", "He asked whether the new schedule would start next week.",
    "To reset the device, hold the power button for five seconds.", "Birds fly south for the winter when the days get shorter.",
    "Q: What color is the sky on a clear day?\nA:", "The bridge was built in the nineteenth century and still carries traffic today.",
    "Check the oil level, then tighten the cap, then record the reading.", "My neighbour grows tomatoes and shares them every summer.",
    "A triangle has three sides and three corners.", "The report is due on Friday; please send corrections by Thursday.",
    "Dogs are loyal animals and many people keep them as pets.", "Turn left at the corner and walk two blocks to reach the station."]


def g12_prompts():
    try:
        from tokenizers import Tokenizer
        tk = Tokenizer.from_pretrained(BASE_REPO)
        out = []
        for t in G12_CANDIDATES:
            ids = tk.encode(t, add_special_tokens=False).ids[:40]
            if len(ids) >= 6 and all(i < 50000 for i in ids):      # only ids that exist in the pruned vocabulary
                out.append(ids)
        if len(out) >= 20:
            return out[:20]
    except Exception as e:  # noqa
        print("g12: HF tokenizer unavailable (", type(e).__name__, "); using random in-vocabulary ids")
    g = random.Random(12)
    return [[g.randrange(1000, 40000) for _ in range(g.randrange(8, 32))] for _ in range(20)]


@torch.no_grad()
def last_and_all_logits(model, prompts, device, autocast_dtype=None):
    outs = []
    for ids in prompts:
        x = torch.tensor([ids], device=device)
        ctx = torch.autocast(device_type=device.type, dtype=autocast_dtype) if autocast_dtype is not None else torch.autocast(device_type=device.type, enabled=False)
        with ctx:
            h = backbone(model)(input_ids=x, use_cache=False).last_hidden_state
            lg = head_logits(model, h, KEPT_REAL)
        outs.append(lg[0].float().cpu())
    return outs


def run_g12():
    prompts = g12_prompts()
    model = AutoModelForCausalLM.from_pretrained(BASE_SRC, config=strip_quantization_config(AutoConfig.from_pretrained(BASE_SRC)), dtype=torch.bfloat16, low_cpu_mem_usage=True)
    model.eval()
    wrap_qat(model)
    ref_dev = DEVICE0
    where = "cpu" if not CUDA else "cuda"
    try:                      # bf16 reference: GPU if bf16 math works there (even emulated on T4), else CPU
        model.to(ref_dev)
        ref = last_and_all_logits(model, prompts, ref_dev, None)
    except Exception as e:  # noqa
        print("bf16 reference on", ref_dev, "failed:", type(e).__name__, str(e)[:100], "-> CPU")
        model.to("cpu")
        ref, where = last_and_all_logits(model, prompts, torch.device("cpu"), None), "cpu"
    # fp32 master weights (exact up-cast of the bf16 master) + fp16 autocast = what training runs
    model.float().to(DEVICE0)
    t16 = torch.float16
    try:
        test = last_and_all_logits(model, prompts, DEVICE0, t16)
        mode16 = "fp16 autocast"
    except Exception as e:  # noqa
        print("fp16 autocast unsupported on this device:", type(e).__name__, str(e)[:100], "-> fp32 test path")
        test, mode16 = last_and_all_logits(model, prompts, DEVICE0, None), "fp32"
    finite = all(bool(torch.isfinite(t).all()) for t in test)
    maxdiff = max(float((a - b).abs().max()) for a, b in zip(ref, test))
    agree = sum(int(a[-1].argmax() == b[-1].argmax()) for a, b in zip(ref, test))
    meandiff = statistics.mean(float((a - b).abs().mean()) for a, b in zip(ref, test))
    res = {"prompts": len(prompts), "reference": f"bf16 params, no autocast, device={where}", "test": f"fp32 master + {mode16}, device={DEVICE0}",
           "finite": finite, "max_abs_logit_diff": maxdiff, "mean_abs_logit_diff": meandiff, "first_token_argmax_agree": agree,
           "gate": "no non-finite values and first-token argmax agreement >= 18 of 20 [plan GATE; no source supplies the 18]",
           "pass": bool(finite and agree >= 18)}
    return model, prompts, res


G12_MODEL, G12_PROMPTS, G12 = run_g12()
print("G12", json.dumps(G12, indent=1))

# ------------------------------------------------------------------ G5 audit: are the five CIS activation-quantization points live in the training forward?
def g5_audit(model, prompts):
    global QAT_PROBE
    QAT_PROBE = {}
    try:
        with torch.no_grad(), torch.autocast(device_type=DEVICE0.type, dtype=AMP_DTYPE if CUDA else torch.bfloat16):
            x = torch.tensor([prompts[0]], device=DEVICE0)
            h = backbone(model)(input_ids=x, use_cache=False).last_hidden_state
            head_logits(model, h, KEPT_REAL)
        probe = dict(QAT_PROBE)
    finally:
        QAT_PROBE = None
    n_layers = model.config.num_hidden_layers
    by_kind = defaultdict(list)
    for k, (err, peak) in probe.items():
        kind = k.split(".")[-1] if not k.startswith("<") else "head"
        by_kind[kind].append((err, peak))
    exp = {"q_proj": n_layers, "k_proj": n_layers, "v_proj": n_layers, "o_proj": n_layers, "gate_proj": n_layers, "up_proj": n_layers, "down_proj": n_layers, "head": 1}
    ok = True
    print("G5 audit (CIS-1 points): input of q/k/v, gate/up (NORMQ output), o_proj (attn_sub_norm output), down_proj (ffn_sub_norm output), head (final-norm output)")
    for kind, n in exp.items():
        v = by_kind.get(kind, [])
        good = len(v) == n and all(e < 2e-3 and abs(p - 127) < 0.05 for e, p in v)
        ok &= good
        print(f"  {kind:10s} quantized inputs seen {len(v):3d}/{n:3d}  max grid error {max((e for e, _ in v), default=float('nan')):.2e}  -> {'on the int8 grid' if good else 'NOT ON GRID'}")
    assert ok, "G5 audit failed: a CIS activation-quantization point is missing from the training forward"
    return {"points_checked": sum(len(v) for v in by_kind.values()), "all_on_int8_grid": ok}


G5 = g5_audit(G12_MODEL, G12_PROMPTS)

# ------------------------------------------------------------------ the QAT layer equals Hugging Face's own online BitLinear (value and straight-through gradient)
def qat_layer_equivalence():
    try:
        from transformers.integrations.bitnet import AutoBitLinear
    except Exception as e:  # noqa
        return {"checked": False, "reason": type(e).__name__}
    torch.manual_seed(1)
    lin = nn.Linear(256, 192, bias=False)
    lin.weight.data.mul_(40.0)
    ref = AutoBitLinear(256, 192, bias=False, online_quant=True)
    ref.weight.data.copy_(lin.weight.data)
    mine = QATLinear(lin, "probe")
    x = (torch.randn(2, 5, 256) * 3).requires_grad_(True)
    x.data[0, 0, 0] = 50.0                                   # an outlier channel, as in the massive-activation layers
    y_ref, y_mine = ref(x), mine(x)
    g = torch.randn_like(y_ref)
    gx_ref, gw_ref = torch.autograd.grad(y_ref, [x, ref.weight], g)
    gx_mine, gw_mine = torch.autograd.grad(y_mine, [x, mine.weight], g)
    return {"checked": True, "max_abs_output_diff": float((y_ref - y_mine).abs().max()), "max_abs_output": float(y_ref.abs().max()),
            "max_abs_dW_diff": float((gw_ref - gw_mine).abs().max()), "max_abs_dX_diff": float((gx_ref - gx_mine).abs().max())}


QEQ = qat_layer_equivalence()
print("QAT layer vs HF AutoBitLinear(online):", json.dumps(QEQ))
if QEQ["checked"]:
    assert QEQ["max_abs_output_diff"] <= 1e-4 * max(1.0, QEQ["max_abs_output"]), "the QAT forward differs from Hugging Face's online BitLinear"
    assert QEQ["max_abs_dW_diff"] <= 1e-5 and QEQ["max_abs_dX_diff"] <= 1e-5, "the straight-through gradient differs from Hugging Face's"

# ------------------------------------------------------------------ real tokens/s of the base forward pass (training dtype, packed sequences)
@torch.no_grad()
def bench_forward(model, n_seq, seq):
    g = torch.Generator().manual_seed(3)
    ids = KEPT_REAL[torch.randint(0, len(KEPT_REAL), (n_seq, seq), generator=g)].to(DEVICE0)
    ctx = lambda: torch.autocast(device_type=DEVICE0.type, dtype=AMP_DTYPE if CUDA else torch.bfloat16)
    with ctx():
        backbone(model)(input_ids=ids[:1], use_cache=False)                       # warm-up
    if CUDA:
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with ctx():
        for i in range(n_seq):
            h = backbone(model)(input_ids=ids[i:i + 1], use_cache=False).last_hidden_state
            head_logits(model, h[:, -16:], KEPT_REAL)
    if CUDA:
        torch.cuda.synchronize()
    return n_seq * seq / (time.perf_counter() - t0)


THROUGHPUT = {}
_tps = bench_forward(G12_MODEL, 8 if CUDA else 2, CFG["seq"])
_gpu = ENV["gpus"][0]["name"] if CUDA else "cpu"
THROUGHPUT["base_forward_tokens_per_s"] = _tps
THROUGHPUT["gpu"] = _gpu
if CUDA:
    free, total = torch.cuda.mem_get_info(0)
    THROUGHPUT["free_mem_gib_after"] = round(free / 2 ** 30, 2)
    THROUGHPUT["peak_mem_gib"] = round(torch.cuda.max_memory_allocated(0) / 2 ** 30, 2)
print(f"E1_THROUGHPUT gpu={_gpu} base_forward tokens_per_s={_tps:.1f} seq={CFG['seq']} dtype={'fp16-autocast' if USE_SCALER else str(AMP_DTYPE)} free_gib={THROUGHPUT.get('free_mem_gib_after', 'n/a')}")

# ------------------------------------------------------------------ informational: does absmean ternarization of the bf16 master reproduce the packed incumbent's trits? (range requests, 3 tensors)
def trit_agreement_vs_packed(names=("model.layers.0.self_attn.q_proj.weight", "model.layers.7.self_attn.k_proj.weight", "model.layers.15.mlp.gate_proj.weight")):
    import urllib.request
    packed_repo = BASE_REPO[:-5] if BASE_REPO.endswith("-bf16") else None
    if SMOKE or packed_repo is None or os.environ.get("E1_CHECK_PACKED", "1") != "1":
        return {"checked": False, "reason": "smoke / no packed sibling repo / disabled"}

    def rng(url, a, b):
        with urllib.request.urlopen(urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}"}), timeout=120) as r:
            return r.read()

    def header(url):
        n = struct.unpack("<Q", rng(url, 0, 7))[0]
        return json.loads(rng(url, 8, 8 + n - 1)), 8 + n
    try:
        ub, up = (f"https://huggingface.co/{r}/resolve/main/model.safetensors" for r in (BASE_REPO, packed_repo))
        hb, bb = header(ub)
        hp, bp = header(up)
        out = []
        for nm in names:
            e, ep = hb[nm], hp[nm]
            w = torch.frombuffer(bytearray(rng(ub, bb + e["data_offsets"][0], bb + e["data_offsets"][1] - 1)), dtype=torch.bfloat16).float().reshape(e["shape"])
            o, i = e["shape"]
            pkd = np.frombuffer(rng(up, bp + ep["data_offsets"][0], bp + ep["data_offsets"][1] - 1), dtype=np.uint8).reshape(o // 4, i)
            q_pk = np.empty((o, i), dtype=np.int8)
            for k in range(4):                       # hf1bitllm packing: field k of byte (r, c) is output row k*(o/4)+r, stored as w+1
                q_pk[k * (o // 4):(k + 1) * (o // 4)] = ((pkd >> (2 * k)) & 3).astype(np.int8) - 1
            q_pk = torch.from_numpy(q_pk)
            g = weight_ternary_gamma(w)
            q = snap_ternary(w, g).to(torch.int8)
            d = q != q_pk
            near = ((w.abs() / g)[d] - 0.5).abs()
            sc = float(torch.frombuffer(bytearray(rng(up, bp + hp[nm + "_scale"]["data_offsets"][0], bp + hp[nm + "_scale"]["data_offsets"][1] - 1)), dtype=torch.bfloat16).float())
            out.append({"tensor": nm, "n": int(q.numel()), "trit_agreement": float((~d).float().mean()), "mismatches": int(d.sum()),
                        "mismatches_within_0.01_of_threshold": float((near < 0.01).float().mean()) if d.any() else None,
                        "mean_abs_w_fp32": float(g), "packed_weight_scale": sc})
        return {"checked": True, "tensors": out,
                "reading": "every mismatch lies within 0.01 of the rounding threshold |w|/gamma = 0.5, i.e. in the bf16 value bin(s) that straddle it; this is consistent with the packed incumbent having been "
                           "quantized from higher-precision latent weights, so the bf16 master cannot reproduce it bit for bit. Step 0 of every arm is a near-incumbent, not the incumbent "
                           "(the Linux-side engine PPL of export/step0 against the incumbent artifact measures the gap). packed weight_scale == bf16(mean|w|) confirms the multiply convention."}
    except Exception as e:  # noqa
        return {"checked": False, "reason": type(e).__name__ + ": " + str(e)[:100]}


TRITS_VS_PACKED = trit_agreement_vs_packed()
print("trit agreement of this notebook's quantizer with the packed incumbent:", json.dumps(TRITS_VS_PACKED, indent=1) if TRITS_VS_PACKED["checked"] else TRITS_VS_PACKED)

del G12_MODEL
gc.collect()
if CUDA:
    torch.cuda.empty_cache()
json.dump({"env": ENV, "quota_before": QUOTA_BEFORE, "g12": G12, "g5": G5, "qat_vs_hf_layer": QEQ, "throughput": THROUGHPUT, "trits_vs_packed_incumbent": TRITS_VS_PACKED}, open(os.path.join(WORK, "cell0.json"), "w"), indent=1)

mode: SMOKE (CPU, tiny random model; numbers are plumbing checks, not measurements)
{
 "mode": "smoke",
 "python": "3.13.16",
 "torch": "2.14.1+cpu",
 "transformers": "5.19.0",
 "base_repo": "microsoft/bitnet-b1.58-2B-4T-bf16",
 "cuda": false,
 "n_gpu": 0,
 "gpus": [],
 "bf16_native": true,
 "amp_dtype": "torch.bfloat16"
}
E1_QUOTA_BEFORE {"status": "not applicable in smoke mode"}


[smoke] tiny random BitNet saved to e1_smoke_out/tiny_base


G12 {
 "prompts": 20,
 "reference": "bf16 params, no autocast, device=cpu",
 "test": "fp32 master + fp16 autocast, device=cpu",
 "finite": true,
 "max_abs_logit_diff": 4.97265625,
 "mean_abs_logit_diff": 0.1320044495165348,
 "first_token_argmax_agree": 19,
 "gate": "no non-finite values and first-token argmax agreement >= 18 of 20 [plan GATE; no source supplies the 18]",
 "pass": true
}
G5 audit (CIS-1 points): input of q/k/v, gate/up (NORMQ output), o_proj (attn_sub_norm output), down_proj (ffn_sub_norm output), head (final-norm output)
  q_proj     quantized inputs seen   2/  2  max grid error 1.53e-05  -> on the int8 grid
  k_proj     quantized inputs seen   2/  2  max grid error 1.53e-05  -> on the int8 grid
  v_proj     quantized inputs seen   2/  2  max grid error 1.53e-05  -> on the int8 grid
  o_proj     quantized inputs seen   2/  2  max grid error 7.63e-06  -> on the int8 grid
  gate_proj  quantized inputs seen   2/  2  max grid error 7.63e-06  -> on the int8 grid
  up_proj  

## Cell 1 - data factory: gateway-verified operator episodes in the `agent_trace` text format

**Episode grammar** (plain text, raw `Q:/A:` form, no special tokens; `encode()` of the engine matches none):

```
<tool-state header>\n
Q: <question>\n
A:[ <call>.]            <- the model's turn (loss)
\nTOOL[<name>]=<out>\n  <- appended by the gateway (`tool_result_text`), NOT model output (no loss)
A: <short operator-voice answer>\n   <- the model's turn (loss)
```

* Every label comes from an exact Python oracle (i64 checked arithmetic, truncating `/`, sign-of-dividend `%`,
  the gateway's fixed strings `div-by-zero` / `overflow`, `NONE` for a LOOKUP miss, `NOT-FOUND` for a FILE-READ miss).
* An episode is **kept only if it round-trips**: the model-turn text is re-parsed by a Python port of the gateway
  scanner (`find_calc`, `find_lookup`, `find_file_read`, `find_tool_call`, `run_tool` from `agent_trace.rs`),
  the re-derived `TOOL[...]` line must equal the oracle's, the verbatim-argument rule must hold, and the final answer
  must carry the tool output. This is a Python port, **not yet differential-tested against the Rust binary**
  (gate G3, >= 1e6 cases, runs on the Linux side); the real `agent_trace verify` is the final authority for receipts.
* Train and dev use **disjoint namespaces** (key formats, prefixes, table names), **disjoint phrasing families**,
  **disjoint header families** and disjoint distractor pools; the dev file carries a canary string that is searched for
  in the training data. Each sample has a `generator` field (G10) and the whole dataset is frozen with a SHA-256 manifest.
* `E1_DATA_DIR` may point at a previously frozen dataset (for example a private Kaggle dataset); its manifest hashes are verified and generation is skipped.

In [2]:
import os, sys, json, math, hashlib, random, re, time, struct, shutil, gc, copy, statistics, itertools
from collections import Counter, defaultdict

# ---------------------------------------------------------------- gateway port (agent_trace.rs semantics)
I64_MIN, I64_MAX = -(2 ** 63), 2 ** 63 - 1
_WS = " \t\n\x0c\r"            # Rust u8::is_ascii_whitespace (no vertical tab)
KEY_OK = set("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789_.-")


def _trunc_div(a, b):
    q = abs(a) // abs(b)
    return -q if (a < 0) != (b < 0) else q


def calc_oracle(a, op, b):
    """Exact string the gateway records as output: decimal result, or 'div-by-zero' / 'overflow'."""
    if op == "+":
        r = a + b
    elif op == "-":
        r = a - b
    elif op == "*":
        r = a * b
    elif op in "/%":
        if b == 0:
            return "div-by-zero"
        if a == I64_MIN and b == -1:
            return "overflow"
        r = _trunc_div(a, b) if op == "/" else a - b * _trunc_div(a, b)
    else:
        raise ValueError(op)
    return str(r) if I64_MIN <= r <= I64_MAX else "overflow"


def is_valid_key(k):
    return 0 < len(k) <= 64 and all(c in KEY_OK for c in k)


def find_calc(text):
    s = text.find("CALC(")
    if s < 0:
        return None
    rest = text[s + 5:]
    c = rest.find(")")
    if c < 0:
        return None
    body, full = rest[:c], text[s:s + 5 + c + 1]
    i = 0

    def skip(i):
        while i < len(body) and body[i] in _WS:
            i += 1
        return i

    def pint(i):
        st = i
        if i < len(body) and body[i] == "-":
            i += 1
        d0 = i
        while i < len(body) and body[i] in "0123456789":
            i += 1
        if i == d0:
            return None
        v = int(body[st:i])
        return (v, i) if I64_MIN <= v <= I64_MAX else None

    i = skip(i)
    r = pint(i)
    if r is None:
        return None
    a, i = r
    i = skip(i)
    if i >= len(body) or body[i] not in "+-*/%":
        return None
    op = body[i]
    i = skip(i + 1)
    r = pint(i)
    if r is None:
        return None
    b, i = r
    if skip(i) != len(body):
        return None
    return full, a, op, b


def _find_keyed(text, opener):
    s = text.find(opener)
    if s < 0:
        return None
    rest = text[s + len(opener):]
    c = rest.find(")")
    if c < 0:
        return None
    key = rest[:c]
    if not is_valid_key(key):
        return None
    return text[s:s + len(opener) + c + 1], key


def find_tool_call(text, table_present):
    cands = []
    for k, op in ((0, "CALC("), (1, "LOOKUP("), (2, "FILE-READ(")):
        if k and not table_present:
            continue
        p = text.find(op)
        if p >= 0:
            cands.append((p, k))
    if not cands:
        return None
    _, k = min(cands)
    if k == 0:
        r = find_calc(text)
        return None if r is None else ("calc",) + r
    r = _find_keyed(text, "LOOKUP(" if k == 1 else "FILE-READ(")
    return None if r is None else (("lookup" if k == 1 else "file-read"),) + r


def run_tool(decoded_text, table, prefix=""):
    """-> (name, input, output) exactly as agent_trace's `run_tool`; table is a dict or None."""
    scanned = prefix + decoded_text
    r = find_tool_call(scanned, table is not None)
    if r is None:
        return "no-tool", "", ""
    if r[0] == "calc":
        _, m, a, op, b = r
        out = calc_oracle(a, op, b)
        return ("calc-error" if out in ("div-by-zero", "overflow") else "calc"), m, out
    _, m, key = r
    if r[0] == "lookup":
        return "lookup", m, table.get(key, "NONE")
    return "file-read", m, table.get(key, "NOT-FOUND")


def tool_result_text(name, out):
    return f"\nTOOL[{name}]={out}\n"


def last_q_line(text):
    q = ""
    for ln in text.split("\n"):
        if ln.startswith("Q:"):
            q = ln[2:].strip()
    return q


def call_arg(inp):
    for p in ("CALC(", "LOOKUP(", "FILE-READ("):
        if inp.startswith(p) and inp.endswith(")"):
            return inp[len(p):-1]
    return None


def verbatim_ok(external, inp):
    a = call_arg(inp)
    return True if not a else a in external


CALL_OPENERS = ("CALC(", "LOOKUP(", "FILE-READ(")


def has_call_opener(text):
    return any(o in text for o in CALL_OPENERS)


# quick self-test of the port against the semantics vectors in test_calc_semantics.py (agent_trace.rs eval_calc)
_vec = [((3, "+", 4), "7"), ((3, "-", 4), "-1"), ((3, "*", 4), "12"), ((7, "/", 2), "3"), ((-7, "/", 2), "-3"),
        ((7, "%", -2), "1"), ((1, "/", 0), "div-by-zero"), ((1, "%", 0), "div-by-zero"), ((I64_MAX, "+", 1), "overflow"),
        ((I64_MIN, "-", 1), "overflow"), ((I64_MIN, "/", -1), "overflow"), ((I64_MIN, "%", -1), "overflow"),
        ((I64_MAX, "*", 2), "overflow"), ((-7, "/", -2), "3"), ((-7, "%", -2), "-1")]
assert all(calc_oracle(*a) == o for a, o in _vec), "oracle self-test failed"
assert find_calc("xx CALC(3 + 4). yy")[1:] == (3, "+", 4) and find_calc("CALC(3+4)")[1:] == (3, "+", 4)
assert find_calc("CALC(3 +)") is None and find_calc("CALC(3 ^ 4)") is None and find_calc("CALC(3 + 4 5)") is None
assert find_calc("CALC(99999999999999999999 + 1)") is None
assert run_tool(" LOOKUP(P-1).", {"P-1": "x"}) == ("lookup", "LOOKUP(P-1)", "x")
assert run_tool(" LOOKUP(P-1).", None)[0] == "no-tool", "LOOKUP is not scanned without a table"
assert run_tool(" FILE-READ(a.txt).", {})[2] == "NOT-FOUND" and run_tool(" LOOKUP(zz).", {})[2] == "NONE"
print("gateway port self-test: OK")

gateway port self-test: OK


In [3]:
# ---------------------------------------------------------------- namespaces, families, pools (train / dev disjoint)
GEN_VERSION = "e1-factory-v1"
CANARY = "E1-CANARY-7f3a91c2-DEV-ONLY"          # embedded in every dev file, must never appear in training data

MATERIALS_T = ["Steel", "Brass", "Nylon", "Copper", "Rubber", "Aluminum", "Zinc", "Titanium", "Bronze", "Teflon"]
MATERIALS_D = ["Cast iron", "Graphite", "Ceramic", "Silicone", "Nickel", "Polymer"]
NOUNS_T = ["bracket", "washer", "valve", "gasket", "bolt", "spring", "filter", "clamp", "hinge", "bearing", "coupling", "rivet", "bushing", "fitting"]
NOUNS_D = ["flange", "sleeve", "cam follower", "manifold", "pinion", "sensor housing"]
SPECS_T = ["M6 x 20 mm", "1/4-20 x 3/4 in.", "10 micron", "left-hand", "right-hand", "OD 32 mm", "grade 5", "zinc plated", "3/8 in.", "heavy duty", "threaded", "sealed"]
SPECS_D = ["ID 18 mm", "type C", "2 per kit", "high temp", "ISO 4762", "matte finish"]
STATUS = ["in stock", "backordered", "discontinued", "on hold", "reserved", "inspected", "shipped", "returned"]
FILE_WORDS_T = ["readme", "notes", "config", "changelog", "schedule", "manifest", "inventory", "report", "roster", "checklist"]
FILE_WORDS_D = ["handover", "runbook", "ledger", "memo", "summary"]
FILE_EXT_T = ["txt", "md", "ini", "log", "cfg"]
FILE_EXT_D = ["csv", "dat", "rpt"]

NS_TRAIN_PREFIXES = ["P", "SKU", "ORD", "INV", "AX", "BN", "CT", "DW", "EQ", "FL", "GR", "HV", "JT", "KP", "LM",
                     "MN", "NR", "OP", "PQ", "RS", "TV", "UW", "VX", "WY", "ZB", "TK", "CRF", "MTL", "SRV", "LOT"]
NS_DEV_PREFIXES = ["qz", "xj", "vk", "wq", "zf", "jx", "kq", "yv"]
PATTERNS_T = ["{p}-{d3}", "{p}{d4}", "{p}_{d5}", "{p}.{d2}.{d3}", "{p}-{l1}{d3}"]
PATTERNS_D = ["{p}-{d2}-{l1}{d2}", "{p}.{l2}{d3}", "{p}_{w}_{d3}"]
ENTITIES_T = ["part", "item", "order", "asset", "ticket", "batch", "record", "unit"]
ENTITIES_D = ["component", "article", "sensor", "line item"]
TABLES_T = ["parts", "orders", "inventory", "assets", "tickets", "batches", "records", "units", "catalog", "ledger"]
TABLES_D = ["shelf", "registry", "depot", "manifest-db"]
RWORDS = ["alder", "birch", "cedar", "dune", "ember", "fjord", "grove", "heath", "inlet", "juniper", "knoll", "larch"]


class NS:
    def __init__(self, idx, split):
        r = random.Random(f"ns-{split}-{idx}")
        self.split, self.idx = split, idx
        self.prefix = (NS_TRAIN_PREFIXES if split == "train" else NS_DEV_PREFIXES)[idx % (30 if split == "train" else 8)]
        self.pattern = r.choice(PATTERNS_T if split == "train" else PATTERNS_D)
        self.entity = r.choice(ENTITIES_T if split == "train" else ENTITIES_D)
        self.table = r.choice(TABLES_T if split == "train" else TABLES_D) + ("" if split == "train" else f"{idx}")
        self.ftable = ("docs" if split == "train" else "vault") + str(idx)
        self.vkind = r.choice(["text", "text", "status", "number"] if split == "train" else ["text", "status", "number"])
        self.fwords = FILE_WORDS_T if split == "train" else FILE_WORDS_D
        self.fext = FILE_EXT_T if split == "train" else FILE_EXT_D
        self.mats, self.nouns, self.specs = ((MATERIALS_T, NOUNS_T, SPECS_T) if split == "train" else (MATERIALS_D, NOUNS_D, SPECS_D))

    def key(self, r):
        up = self.split == "train"
        return self.pattern.format(
            p=self.prefix, d2="%02d" % r.randrange(100), d3="%03d" % r.randrange(1000), d4="%04d" % r.randrange(10000),
            d5="%05d" % r.randrange(100000), l1=r.choice("ABCDEFGHJKLMNPRSTUVWXYZ" if up else "abcdefghjkmnpqrstuvwxyz"),
            l2="".join(r.choice("abcdefghjkmnpqrstuvwxyz") for _ in range(2)), w=r.choice(RWORDS))

    def value(self, r):
        if self.vkind == "text":
            return f"{r.choice(self.mats)} {r.choice(self.nouns)}, {r.choice(self.specs)}"
        if self.vkind == "status":
            return f"{r.choice(STATUS)}, {r.randrange(1, 400)} units, bin {r.choice('ABCDEFG')}{r.randrange(1, 40)}"
        return str(r.randrange(1, 100000))

    def fname(self, r):
        w = r.choice(self.fwords)
        if self.split == "train":
            return f"{w}.{r.choice(self.fext)}" if r.random() < .6 else f"{w}_{r.randrange(1, 100):02d}.{r.choice(self.fext)}"
        return f"{self.prefix}-{w}_{r.randrange(10000):04d}.{r.choice(self.fext)}"   # dev tables are merged into one gateway table: names must not collide across worlds

    def fcontent(self, r):
        return r.choice(["Service interval is %d hours.", "Last checked on shift %d.", "Quota for the week is %d units.",
                         "Contact extension is %d.", "Revision %d approved.", "Open items: %d."]) % r.randrange(2, 900)


NS_TRAIN = [NS(i, "train") for i in range(30)]
NS_DEV = [NS(i, "dev") for i in range(8)]
assert not ({n.prefix for n in NS_TRAIN} & {n.prefix for n in NS_DEV})
assert not ({n.table for n in NS_TRAIN} & {n.table for n in NS_DEV})

# header families: a tool exists only when it is declared (G2). Plain text, one line, ZERO example calls (condition P0).
HEADERS_T = [("TOOLS: {l}", "TOOLS: none"), ("Available tools: {l}.", "Available tools: none."),
             ("[tools] {l}", "[tools] (none)"), ("You may call: {l}.", "You may call no tools.")]
HEADERS_D = [("Tool state -> {l}", "Tool state -> empty"), ("Declared: {l}", "Declared: nothing")]

# phrasing families (train vs held-out dev)
Q_HIT_T = ["Look up {k}.", "Find {k} in the {t} table.", "Check the {t} table for {k}.", "{e} {k}", "Get the record for {k}."]
Q_HIT_D = ["Pull up {k}.", "Retrieve {k} from {t}.", "{k} -- details?"]
Q_UNK_T = ["What is the value recorded for {e} {k}?", "Tell me what {e} {k} is.", "What is listed for {k}?",
           "Describe {e} {k}.", "What does the record say about {k}?", "I need the description of {e} {k}."]
Q_UNK_D = ["Can you tell me about {e} {k}?", "What's {k}?", "Which {e} is {k}?", "Give me the details on {k}, please.",
           "Quick question: what is {e} {k}?"]
Q_FILE_T = ["Read the file {k}.", "Open {k} and tell me what it says.", "What does the file {k} contain?", "FILE {k}"]
Q_FILE_D = ["Show me {k}.", "Print the contents of {k}.", "What is in {k}?"]
Q_CALC_T = ["{x}", "{x}", "{x}", "What is {x}?", "Compute {x}.", "Calculate {x}", "Work out {x}"]
Q_CALC_D = ["{x} = ?", "Evaluate {x}", "How much is {x}?"]
Q_MIX_T = ["Compute {x}, then look up {k}.", "{x}; also {k}?", "Two things: {x} and the record for {k}."]
Q_MIX_D = ["First {x}, afterwards {k} please.", "Need {x} and also {k}."]
Q_MIX_REV_T = ["Look up {k}, then compute {x}.", "{k}? and also {x}", "Two things: the record for {k} and {x}."]
Q_MIX_REV_D = ["First {k}, afterwards {x} please.", "Need {k} and also {x}."]

# short operator-voice answers
A_CALC_T = ["{x} = {r}.", "The result is {r}.", "{r}.", "{x} comes to {r}.", "That is {r}."]
A_CALC_D = ["Answer: {r}.", "{x} equals {r}."]
A_ERR_DIV = ["That cannot be computed: division by zero.", "Division by zero. No result."]
A_ERR_OVF = ["That result is outside the 64-bit integer range (overflow).", "Overflow: the result does not fit in 64 bits."]
A_HIT_T = ["{k}: {v}", "The table lists {k} as {v}", "{v} ({k}).", "Record {k} reads: {v}"]
A_HIT_D = ["For {k} I have: {v}", "{k} is {v}"]
A_NONE_T = ["No entry for {k} in the table.", "{k} is not in the table.", "Not found: {k}."]
A_NONE_D = ["The table has no record for {k}.", "I find nothing under {k}."]
A_FHIT_T = ["{k} says: {v}", "{k}: {v}"]
A_FHIT_D = ["The file {k} reads: {v}"]
A_FNF_T = ["File {k} not found.", "{k} is not in the file store."]
A_FNF_D = ["There is no file called {k}."]
A_NOCALC = ["No calculator is declared, so I cannot compute that reliably.", "I have no calculator declared here. Supply one or compute it elsewhere."]
A_NOLOOK = ["No lookup table is declared, so I cannot check that.", "That is not in my data and no table is declared. Supply a table or ask a connected unit."]
A_NOFILE = ["No file access is declared, so I cannot read {k}.", "I cannot open {k}: no file tool is declared."]
A_NOCALC_D = ["Without a declared calculator I will not guess that sum."]
A_NOLOOK_D = ["I do not have that, and no table is declared to look it up."]
A_NOFILE_D = ["File access is not declared, so {k} stays closed."]

DIST_T = [
    ("What is the capital of France?", "Paris."), ("What is the capital of Japan?", "Tokyo."), ("What is the capital of Italy?", "Rome."),
    ("What is the capital of Canada?", "Ottawa."), ("What is the capital of Egypt?", "Cairo."), ("What is the capital of Spain?", "Madrid."),
    ("How many days are in a week?", "Seven."), ("How many months are in a year?", "Twelve."), ("How many hours are in a day?", "Twenty-four."),
    ("How many minutes are in an hour?", "Sixty."), ("How many legs does a spider have?", "Eight."), ("How many sides does a triangle have?", "Three."),
    ("How many sides does a hexagon have?", "Six."), ("How many continents are there?", "Seven."), ("How many wheels does a bicycle have?", "Two."),
    ("What color is the sky on a clear day?", "Blue."), ("What color do you get by mixing blue and yellow?", "Green."), ("What color is a ripe banana?", "Yellow."),
    ("What do cows give us to drink?", "Milk."), ("What do bees make?", "Honey."), ("What do you call a baby dog?", "A puppy."),
    ("What do you call a baby cat?", "A kitten."), ("What is frozen water called?", "Ice."), ("What gas do plants absorb from the air?", "Carbon dioxide."),
    ("What gas do humans need to breathe?", "Oxygen."), ("What is the largest planet in the solar system?", "Jupiter."),
    ("Which planet is closest to the sun?", "Mercury."), ("Which planet do we live on?", "Earth."), ("What is the chemical symbol for water?", "H2O."),
    ("What is the chemical symbol for gold?", "Au."), ("What is the boiling point of water in Celsius at sea level?", "One hundred degrees."),
    ("What is the freezing point of water in Celsius?", "Zero degrees."), ("Which ocean is the largest?", "The Pacific Ocean."),
    ("What is the opposite of hot?", "Cold."), ("What is the opposite of up?", "Down."), ("What is the opposite of fast?", "Slow."),
    ("What is the plural of mouse?", "Mice."), ("What is the plural of child?", "Children."), ("What is the past tense of run?", "Ran."),
    ("What day comes after Monday?", "Tuesday."), ("What day comes after Friday?", "Saturday."), ("What month comes after March?", "April."),
    ("What month comes before June?", "May."), ("What season comes after winter?", "Spring."), ("Who wrote Romeo and Juliet?", "William Shakespeare."),
    ("What language is spoken in Brazil?", "Portuguese."), ("What language is spoken in Mexico?", "Spanish."), ("What currency is used in Japan?", "The yen."),
    ("What is the first letter of the alphabet?", "A."), ("What is the last letter of the alphabet?", "Z."), ("Spell the word cat.", "C-A-T."),
    ("Spell the word sun.", "S-U-N."), ("Say hello.", "Hello."), ("Say thank you.", "Thank you."), ("What does a thermometer measure?", "Temperature."),
    ("What does a clock show?", "The time."), ("What tool do you use to hammer a nail?", "A hammer."), ("What do we call a person who flies a plane?", "A pilot."),
    ("What animal is known as the king of the jungle?", "The lion."), ("What is the tallest animal?", "The giraffe."), ("What is the largest mammal?", "The blue whale."),
    ("What shape is a ball?", "A sphere."), ("What shape has four equal sides?", "A square."), ("Is a tomato a fruit or a vegetable?", "Botanically, a fruit."),
    ("Is the sun a star?", "Yes."), ("Is a whale a fish?", "No, it is a mammal."), ("Does water flow uphill by itself?", "No."),
    ("Which is heavier, a kilogram of feathers or a kilogram of iron?", "They weigh the same."), ("What is the square root of 81?", "Nine."),
    ("What is half of ten?", "Five."), ("What is double four?", "Eight."), ("What is the next number after nine?", "Ten."),
    ("What comes after the number nineteen?", "Twenty."), ("What is the shape of a stop sign?", "An octagon."), ("What is a group of wolves called?", "A pack."),
    ("What is the main ingredient of bread?", "Flour."), ("What do we use to cut paper?", "Scissors."), ("What is the study of living things called?", "Biology."),
    ("What is the process of water turning into vapor called?", "Evaporation."), ("What force pulls objects toward the earth?", "Gravity."),
    ("What is the largest desert on earth?", "The Antarctic Desert."), ("What is the longest river in Africa?", "The Nile."),
    ("Who painted the Mona Lisa?", "Leonardo da Vinci."), ("What instrument has black and white keys?", "The piano."),
]
DIST_D = [
    ("Which country is Berlin the capital of?", "Germany."), ("What is the capital of Australia?", "Canberra."), ("How many players are on a soccer team on the field?", "Eleven."),
    ("How many strings does a standard guitar have?", "Six."), ("What is the color of grass?", "Green."), ("What do hens lay?", "Eggs."),
    ("What is the plural of tooth?", "Teeth."), ("What is the past tense of go?", "Went."), ("What is the smallest prime number?", "Two."),
    ("How many seconds are in a minute?", "Sixty."), ("What is the symbol for the element oxygen?", "O."), ("What do you call a baby cow?", "A calf."),
    ("What is the opposite of open?", "Closed."), ("Which direction does the sun rise from?", "The east."), ("What is the hardest natural substance?", "Diamond."),
    ("What instrument has strings and a bow?", "The violin."), ("What do we call frozen rain?", "Hail."), ("Name a primary color.", "Red."),
    ("How many zeros are in one hundred?", "Two."), ("What is the shape of a dice?", "A cube."), ("Say good morning.", "Good morning."),
    ("What is the largest ocean animal?", "The blue whale."), ("What is the second month of the year?", "February."), ("Spell the word dog.", "D-O-G."),
    ("What is the currency of the United Kingdom?", "The pound."), ("What do you wear on your feet?", "Shoes."), ("Which metal is liquid at room temperature?", "Mercury."),
    ("How many days are in a leap year?", "Three hundred sixty-six."), ("What is the freezing point of water in Fahrenheit?", "Thirty-two degrees."),
    ("Who wrote Hamlet?", "William Shakespeare."), ("What is the tallest mountain on earth?", "Mount Everest."), ("What planet is known as the red planet?", "Mars."),
]
NUMW = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine"]
NUMW_CAP = ["Zero", "One", "Two", "Three", "Four", "Five", "Six", "Seven", "Eight", "Nine"]
UNANS_T = [("What is the name of my neighbor's dog?", "I do not know. That is not in my data."),
           ("What did I eat for breakfast yesterday?", "I do not know. I have no record of that."),
           ("What will the weather be next month?", "I cannot know that. I have no data for it."),
           ("Who will win the next election?", "I cannot know that. It is not in my data."),
           ("What is the password of the front door?", "I do not have that, and I should not guess it."),
           ("How tall is the person at the next desk?", "I do not know. That is not in my data."),
           ("What number am I thinking of?", "I cannot know that."),
           ("What was said in the meeting I missed?", "I do not know. I have no minutes of it."),
           ("Which bus will be late tomorrow?", "I cannot know that. No schedule data is declared."),
           ("What is my bank balance?", "I do not have that. No account data is declared."),
           ("What is the serial number of the machine in the other building?", "I do not have that in my data."),
           ("What time did the delivery truck leave this morning?", "I do not know. I have no log of it.")]
UNANS_D = [("What is the phone number of my dentist?", "I do not have that in my data."),
           ("Which horse wins the race on Sunday?", "I cannot know that."),
           ("What is written on the note on my desk?", "I do not know. I cannot see it."),
           ("How many people are in the lobby right now?", "I do not know. I have no sensor data declared.")]
THINGS = ["crate", "pallet", "motor", "drum", "panel", "cart", "valve body", "spool"]
PEOPLE = ["Ana", "Ben", "Carla", "Dev", "Elena", "Farid", "Gita", "Hugo", "Ines", "Jonas", "Kira", "Leo"]
CODEWORDS = ["amber", "falcon", "marble", "harbor", "violet", "copper", "meadow", "lantern"]


def _wrap_header(r, declared, split):
    fam = HEADERS_T if split == "train" else HEADERS_D
    tpl, none = r.choice(fam)
    return none if not declared else tpl.format(l=", ".join(declared))


def _declare(r, ns, need, extra=True):
    """Declared tool list: always the needed tools, plus random extras (a tool exists only when declared)."""
    d = set(need)
    if extra:
        for t, p in (("CALC", .40), ("LOOKUP", .30), ("FILE-READ", .20)):
            if r.random() < p:
                d.add(t)
    names = []
    if "CALC" in d:
        names.append("CALC")
    if "LOOKUP" in d:
        names.append(f"LOOKUP:{ns.table}")
    if "FILE-READ" in d:
        names.append(f"FILE-READ:{ns.ftable}")
    return d, names


def _world(r, ns, kind="lookup", n=None):
    n = n or r.randrange(6, 14)
    t = {}
    while len(t) < n:
        if kind == "lookup":
            k = ns.key(r)
            t[k] = ns.value(r)
        else:
            k = ns.fname(r)
            t[k] = ns.fcontent(r)
    return t


def _mutate_key(r, key, table):
    for _ in range(50):
        c = list(key)
        m = r.randrange(5)
        if m == 0 and any(ch.isalpha() for ch in key):
            i = r.choice([i for i, ch in enumerate(c) if ch.isalpha()])
            c[i] = c[i].swapcase()
        elif m == 1 and any(ch.isdigit() for ch in key):
            i = r.choice([i for i, ch in enumerate(c) if ch.isdigit()])
            c[i] = str((int(c[i]) + r.randrange(1, 9)) % 10)
        elif m == 2:
            c.append(r.choice("0123456789Xx"))
        elif m == 3 and len(c) > 3:
            del c[r.randrange(len(c))]
        else:
            if len(c) > 3:
                i = r.randrange(len(c) - 1)
                c[i], c[i + 1] = c[i + 1], c[i]
        k2 = "".join(c)
        if k2 != key and k2 not in table and is_valid_key(k2):
            return k2
    return key + "9"


def _fresh_key(r, ns, table, files=False):
    for _ in range(100):
        k = ns.fname(r) if files else ns.key(r)
        if k not in table:
            return k
    raise RuntimeError("could not draw a fresh key")


def _rand_int(r, lo, hi):
    return r.randint(lo, hi)


def _expr(a, op, b, tight=False):
    return f"{a}{op}{b}" if tight else f"{a} {op} {b}"


def _ep(gen, bucket, split, ns, declared, segs, meta=None):
    return {"generator": gen, "bucket": bucket, "split": split, "ns": ns.idx if ns else -1, "declared": sorted(declared),
            "segs": [list(s) for s in segs], "meta": meta or {}}


def _calc_final(r, split, a, op, b, out, tight=False):
    x = _expr(a, op, b, tight)
    if out == "div-by-zero":
        return r.choice(A_ERR_DIV)
    if out == "overflow":
        return r.choice(A_ERR_OVF)
    return r.choice(A_CALC_T if split == "train" else A_CALC_D).format(x=x, r=out)


def _calc_episode(r, split, ns, gen, bucket, a, op, b):
    tight = r.random() < .10
    x = _expr(a, op, b, tight)
    need = {"CALC"}
    dec, names = _declare(r, ns, need)
    out = calc_oracle(a, op, b)
    q = r.choice(Q_CALC_T if split == "train" else Q_CALC_D).format(x=x)
    name = "calc-error" if out in ("div-by-zero", "overflow") else "calc"
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:"),
            ("gen", f" CALC({x})."),
            ("tool", tool_result_text(name, out)),
            ("gen", "A: " + _calc_final(r, split, a, op, b, out, tight) + "\n")]
    return _ep(gen, bucket, split, ns, dec, segs, {"calls": [f"CALC({x})"], "outputs": [out], "table": None})


def g_calc_easy(r, split):
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    return _calc_episode(r, split, ns, "calc_easy", "calc_easy", _rand_int(r, 0, 20), r.choice("+-*"), _rand_int(r, 0, 20))


def g_calc_hard(r, split):
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    op = r.choice("+-*/%")
    if r.random() < .25 and op == "*":
        a, b = _rand_int(r, 100, 99999), _rand_int(r, 100, 99999)
    else:
        a, b = _rand_int(r, 21, 9999), _rand_int(r, 21, 9999)
    if r.random() < .08:
        a = -a
    if op in "/%" and b == 0:
        b = 7
    return _calc_episode(r, split, ns, "calc_hard", "calc_hard", a, op, b)


def g_calc_overflow(r, split):
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    m = r.random()
    if m < .22:                                   # division / remainder by zero -> fixed error string
        a, op, b = _rand_int(r, -10 ** 6, 10 ** 6), r.choice("/%"), 0
    elif m < .40:
        a, op, b = I64_MAX - _rand_int(r, 0, 5), "+", _rand_int(r, 1, 50)
    elif m < .52:
        a, op, b = I64_MIN + _rand_int(r, 0, 5), "-", _rand_int(r, 1, 50)
    elif m < .64:
        a, op, b = r.choice([I64_MIN, I64_MIN + 1]), r.choice("/%"), -1
    elif m < .82:
        a, op, b = _rand_int(r, 10 ** 9, 10 ** 12), "*", _rand_int(r, 10 ** 9, 10 ** 12)
    else:                                         # large but in range
        a, op, b = _rand_int(r, 10 ** 4, 99999), "*", _rand_int(r, 10 ** 4, 99999)
    return _calc_episode(r, split, ns, "calc_overflow", "calc_overflow", a, op, b)


def _lookup_episode(r, split, gen, bucket, mode, phrasing):
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    table = _world(r, ns)
    real = r.choice(sorted(table))
    if mode == "hit":
        key = real
    elif mode == "miss":
        key = _fresh_key(r, ns, table)
    else:
        key = _mutate_key(r, real, table)
    dec, names = _declare(r, ns, {"LOOKUP"})
    q = r.choice(phrasing).format(k=key, t=ns.table, e=ns.entity)
    out = table.get(key, "NONE")
    ans = (r.choice(A_HIT_T if split == "train" else A_HIT_D).format(k=key, v=out) if key in table
           else r.choice(A_NONE_T if split == "train" else A_NONE_D).format(k=key))
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:"), ("gen", f" LOOKUP({key})."),
            ("tool", tool_result_text("lookup", out)), ("gen", "A: " + ans + "\n")]
    return _ep(gen, bucket, split, ns, dec, segs, {"calls": [f"LOOKUP({key})"], "outputs": [out], "table": table, "key": key})


def g_lookup_hit(r, split):
    return _lookup_episode(r, split, "lookup_hit", "lookup_hit", "hit", Q_HIT_T if split == "train" else Q_HIT_D)


def g_lookup_miss(r, split):
    return _lookup_episode(r, split, "lookup_miss", "lookup_miss", "miss", Q_HIT_T if split == "train" else Q_HIT_D)


def g_lookup_near(r, split):
    return _lookup_episode(r, split, "lookup_near_miss", "lookup_near_miss", "near", Q_HIT_T if split == "train" else Q_HIT_D)


def g_unknown_fact(r, split):
    """Plain question (no tool cue) whose answer sits in a declared table, never in the weights: must produce LOOKUP(<verbatim key>)."""
    mode = "hit" if r.random() < .85 else "miss"
    return _lookup_episode(r, split, "unknown_fact", "unknown_fact", mode, Q_UNK_T if split == "train" else Q_UNK_D)


def g_file_read(r, split):
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    table = _world(r, ns, "file")
    hit = r.random() < .7
    key = r.choice(sorted(table)) if hit else _fresh_key(r, ns, table, files=True)
    dec, names = _declare(r, ns, {"FILE-READ"})
    q = r.choice(Q_FILE_T if split == "train" else Q_FILE_D).format(k=key)
    out = table.get(key, "NOT-FOUND")
    ans = (r.choice(A_FHIT_T if split == "train" else A_FHIT_D).format(k=key, v=out) if hit
           else r.choice(A_FNF_T if split == "train" else A_FNF_D).format(k=key))
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:"), ("gen", f" FILE-READ({key})."),
            ("tool", tool_result_text("file-read", out)), ("gen", "A: " + ans + "\n")]
    return _ep("file_read", "file_read", split, ns, dec, segs, {"calls": [f"FILE-READ({key})"], "outputs": [out], "table": table, "key": key})


def g_mixed(r, split):
    """Two tool calls in one answer; both arguments appear verbatim in the Q line."""
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    table = _world(r, ns)
    key = r.choice(sorted(table))
    a, op, b = _rand_int(r, 0, 99), r.choice("+-*"), _rand_int(r, 0, 99)
    x = _expr(a, op, b)
    cout = calc_oracle(a, op, b)
    lout = table[key]
    calc_first = r.random() < .5
    fam = ((Q_MIX_T if calc_first else Q_MIX_REV_T) if split == "train" else (Q_MIX_D if calc_first else Q_MIX_REV_D))
    q = r.choice(fam).format(x=x, k=key)
    dec, names = _declare(r, ns, {"CALC", "LOOKUP"})
    c_seg = [("gen", f" CALC({x})."), ("tool", tool_result_text("calc", cout))]
    l_seg = [("gen", f" LOOKUP({key})."), ("tool", tool_result_text("lookup", lout))]
    first, second = (c_seg, l_seg) if calc_first else (l_seg, c_seg)
    second = [("gen", "A:" + second[0][1]), second[1]]
    ans = (f"{x} = {cout}; {key}: {lout}" if calc_first else f"{key}: {lout}; {x} = {cout}")
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:")] + first + second + [("gen", "A: " + ans + "\n")]
    calls = [f"CALC({x})", f"LOOKUP({key})"] if calc_first else [f"LOOKUP({key})", f"CALC({x})"]
    outs = [cout, lout] if calc_first else [lout, cout]
    return _ep("mixed", "mixed", split, ns, dec, segs, {"calls": calls, "outputs": outs, "table": table, "key": key})


def g_distractor(r, split):
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    dec, names = _declare(r, ns, set())
    if r.random() < .15:                           # CALC-shaped but non-integer operands: no tool
        a, b = r.randrange(1, 9), r.randrange(1, 9)
        form = r.choice(["{A} + {B}", "{A} plus {B}"] if split == "train" else ["{A} and {B} added", "{A} plus {B} equals?"])
        q = form.format(A=NUMW[a], B=NUMW[b])
        ans = (NUMW_CAP[a + b] if a + b < 10 else {10: "Ten", 11: "Eleven", 12: "Twelve", 13: "Thirteen", 14: "Fourteen", 15: "Fifteen", 16: "Sixteen"}[a + b]) + "."
    else:
        q, ans = r.choice(DIST_T if split == "train" else DIST_D)
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:"), ("gen", " " + ans + "\n")]
    return _ep("distractor", "distractor", split, ns, dec, segs, {"calls": [], "outputs": [], "table": None})


def g_in_context(r, split):
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    dec, names = _declare(r, ns, set())
    n = r.randrange(2, 900)
    th = r.choice(THINGS)
    if split == "train":
        kind = r.randrange(4)
        if kind == 0:
            q, ans = f"The {th} weighs {n} kg. How much does the {th} weigh?", f"{n} kg."
        elif kind == 1:
            d = f"2026-{r.randrange(1, 13):02d}-{r.randrange(1, 29):02d}"
            q, ans = f"Pump {n} was serviced on {d}. When was pump {n} serviced?", f"{d}."
        elif kind == 2:
            hh, mm = r.randrange(5, 22), r.choice(["00", "15", "30", "45"])
            q, ans = f"Shift {r.choice('ABC')} starts at {hh}:{mm}. What time does the shift start?", f"{hh}:{mm}."
        else:
            p = r.choice(PEOPLE)
            q, ans = f"{p} is the supervisor on duty. Who is the supervisor on duty?", f"{p}."
    else:
        kind = r.randrange(3)
        if kind == 0:
            q, ans = f"Note: the {th} measures {n} cm. What is the measured size of the {th}?", f"{n} cm."
        elif kind == 1:
            w = r.choice(CODEWORDS)
            q, ans = f"Today's code word is {w}. What is the code word?", f"{w}."
        else:
            p = r.choice(PEOPLE)
            q, ans = f"Dock {n} is assigned to {p}. Who has dock {n}?", f"{p}."
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:"), ("gen", " " + ans + "\n")]
    return _ep("in_context_answerable", "in_context_answerable", split, ns, dec, segs, {"calls": [], "outputs": [], "table": None})


def g_undeclared(r, split):
    """The question needs a tool that the header does not declare: abstain with the reason, never call."""
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    need = r.choice(["CALC", "LOOKUP", "FILE-READ"])
    others = [t for t in ("CALC", "LOOKUP", "FILE-READ") if t != need]
    declared = {t for t in others if r.random() < .5}
    names = [("CALC" if t == "CALC" else f"LOOKUP:{ns.table}" if t == "LOOKUP" else f"FILE-READ:{ns.ftable}") for t in ("CALC", "LOOKUP", "FILE-READ") if t in declared]
    d = split == "dev"
    if need == "CALC":
        a, op, b = _rand_int(r, 21, 9999), r.choice("+-*"), _rand_int(r, 21, 9999)
        q = r.choice(Q_CALC_T if not d else Q_CALC_D).format(x=_expr(a, op, b))
        ans = r.choice(A_NOCALC_D if d else A_NOCALC)
    elif need == "LOOKUP":
        key = ns.key(r)
        q = r.choice(Q_UNK_D if d else Q_UNK_T).format(k=key, e=ns.entity)
        ans = r.choice(A_NOLOOK_D if d else A_NOLOOK)
    else:
        key = ns.fname(r)
        q = r.choice(Q_FILE_D if d else Q_FILE_T).format(k=key)
        ans = r.choice(A_NOFILE_D if d else A_NOFILE).format(k=key)
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:"), ("gen", " " + ans + "\n")]
    return _ep("undeclared_tool", "undeclared_tool", split, ns, declared, segs, {"calls": [], "outputs": [], "table": None, "needs": need})


ATTR_T = ["phone number", "home address", "locker code", "badge number", "shoe size", "favorite color", "birthday", "middle name"]
ATTR_D = ["license plate", "salary", "email password", "seat number"]
EVENTS_T = ["the next election", "tomorrow's match", "the lottery", "the cup final", "the regional contest"]
EVENTS_D = ["the chess open", "Sunday's race"]
WHENS_T = ["next week", "tomorrow", "on Friday", "next month"]
WHENS_D = ["this evening", "in March"]
MEETS_T = ["meeting", "briefing", "review", "stand-up"]
MEETS_D = ["handover", "site walk"]


def g_unanswerable(r, split):
    """Open questions that no declared tool or table covers: say so plainly (abstain with reason), never guess."""
    ns = r.choice(NS_TRAIN if split == "train" else NS_DEV)
    dec, names = _declare(r, ns, set(), extra=False)
    t = split == "train"
    if r.random() < .25:
        q, ans = r.choice(UNANS_T if t else UNANS_D)
    else:
        m = r.randrange(4 if t else 3)
        p = r.choice(PEOPLE)
        if m == 0:
            q, ans = f"What is {p}'s {r.choice(ATTR_T if t else ATTR_D)}?", r.choice(["I do not have that in my data.", "I do not know. That is not in my data."])
        elif m == 1:
            q, ans = f"What did {p} say at the {r.choice(MEETS_T if t else MEETS_D)}?", "I do not know. I have no record of it."
        elif m == 2:
            q, ans = f"Who will win {r.choice(EVENTS_T if t else EVENTS_D)}?", "I cannot know that. It is not in my data."
        else:
            q, ans = f"How many {r.choice(THINGS)}s will arrive {r.choice(WHENS_T if t else WHENS_D)}?", "I cannot know that. No schedule data is declared."
    segs = [("ctx", _wrap_header(r, names, split) + f"\nQ: {q}\nA:"), ("gen", " " + ans + "\n")]
    return _ep("unanswerable_open", "unanswerable", split, ns, dec, segs, {"calls": [], "outputs": [], "table": None})


GENERATORS = {   # name -> (fn, mix weight in %). Tool-requiring about 70%, no-call about 30% (floors: >= 10% no-tool, >= 10% unanswerable)
    "calc_easy": (g_calc_easy, 10), "calc_hard": (g_calc_hard, 12), "calc_overflow": (g_calc_overflow, 8),
    "lookup_hit": (g_lookup_hit, 8), "lookup_miss": (g_lookup_miss, 4), "lookup_near_miss": (g_lookup_near, 4),
    "file_read": (g_file_read, 6), "unknown_fact": (g_unknown_fact, 12), "mixed": (g_mixed, 6),
    "distractor": (g_distractor, 8), "in_context_answerable": (g_in_context, 6), "undeclared_tool": (g_undeclared, 10),
    "unanswerable": (g_unanswerable, 6),
}
assert sum(w for _, w in GENERATORS.values()) == 100
print("generators:", {k: w for k, (_, w) in GENERATORS.items()})

generators: {'calc_easy': 10, 'calc_hard': 12, 'calc_overflow': 8, 'lookup_hit': 8, 'lookup_miss': 4, 'lookup_near_miss': 4, 'file_read': 6, 'unknown_fact': 12, 'mixed': 6, 'distractor': 8, 'in_context_answerable': 6, 'undeclared_tool': 10, 'unanswerable': 6}


In [4]:
# ---------------------------------------------------------------- round-trip verification, corpus build, freeze
def episode_text(ep):
    return "".join(t for _, t in ep["segs"])


def roundtrip(ep):
    """generate -> execute (gateway port) -> TOOL line -> verify. Returns None if the episode round-trips, else a reason."""
    segs, meta = ep["segs"], ep["meta"]
    table = meta.get("table")
    text = episode_text(ep)
    if not text.isascii() or "\t" in text:
        return "non-ascii or tab"
    calls, outs, seen, external = [], [], "", ""
    for i, (kind, txt) in enumerate(segs):
        if kind == "ctx":
            seen += txt
            external = last_q_line(seen)
        elif kind == "gen":
            name, inp, out = run_tool(txt, table)
            nxt = segs[i + 1] if i + 1 < len(segs) else None
            if nxt is not None and nxt[0] == "tool":
                if name == "no-tool":
                    return "model turn has no parseable call but is followed by a TOOL line"
                if nxt[1] != tool_result_text(name, out):
                    return f"oracle TOOL line {nxt[1]!r} != gateway {tool_result_text(name, out)!r}"
                if not verbatim_ok(external, inp):
                    return f"argument not verbatim in context: {inp}"
                calls.append(inp)
                outs.append(out)
                external += nxt[1]
            else:
                if name != "no-tool" or has_call_opener(txt):
                    return "final/no-call turn contains a call opener"
            seen += txt
        else:
            seen += txt
    if calls != meta["calls"] or outs != meta["outputs"]:
        return "calls/outputs differ from generator metadata"
    final = segs[-1][1] if segs[-1][0] == "gen" else ""
    if not final.endswith("\n"):
        return "last turn does not end with newline"
    for out in outs:
        if out in ("div-by-zero", "overflow"):
            if ("zero" if out == "div-by-zero" else "overflow") not in final.lower():
                return "error output not reflected in the answer"
        elif out not in ("NONE", "NOT-FOUND") and out not in final:
            return "tool output missing from the final answer"
    if meta.get("key") and meta["key"] not in final and ep["bucket"] in ("lookup_hit", "lookup_miss", "lookup_near_miss", "unknown_fact", "file_read", "mixed"):
        return "key missing from the final answer"
    return None


REPEAT_CAP = 2     # finite pools (distractor facts, abstention prompts) may repeat an exact text at most this many times


def build_episodes(rng, split, plan, max_tries=50, seen=None):
    """plan: list of generator names to emit, in order. Returns (episodes, stats). Only round-trip failures count against survival."""
    out, stats = [], Counter()
    seen = Counter() if seen is None else seen
    for gname in plan:
        fn = GENERATORS[gname][0]
        for _ in range(max_tries):
            ep = fn(rng, split)
            stats["generated"] += 1
            why = roundtrip(ep)
            if why:
                stats["rejected"] += 1
                stats["rejected:" + why[:40]] += 1
                continue
            h = hashlib.sha1(episode_text(ep).encode()).digest()
            if seen[h] >= REPEAT_CAP:
                stats["duplicate"] += 1
                continue
            seen[h] += 1
            out.append(ep)
            stats["kept"] += 1
            break
        else:
            stats["skipped_saturated:" + gname] += 1
    return out, stats


def weighted_plan(rng, n):
    names = list(GENERATORS)
    w = [GENERATORS[k][1] for k in names]
    return rng.choices(names, weights=w, k=n)


def jsonable(ep, keep_table):
    m = dict(ep["meta"])
    if not keep_table:
        m.pop("table", None)
    d = dict(ep)
    d["meta"] = m
    return d


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def write_jsonl(path, eps, keep_table, tag):
    with open(path, "w") as f:
        for i, ep in enumerate(eps):
            d = jsonable(ep, keep_table)
            d["id"] = f"{tag}-{i:07d}"
            if tag == "dev":
                d["canary"] = CANARY
            f.write(json.dumps(d, separators=(",", ":"), ensure_ascii=True) + "\n")


DATA_DIR = os.path.join(WORK, "data")
os.makedirs(DATA_DIR, exist_ok=True)
SEED = int(os.environ.get("E1_SEED", "20261007"))
EXTERNAL_DATA = os.environ.get("E1_DATA_DIR", "")
BYTES_PER_TOKEN_GUESS = 2.4     # measured on the engine-port tokenization of these episodes; conservative; Cell 2 measures the real figure with the engine-port tokenizer and trims or extends

TRAIN_RNG = random.Random(f"{SEED}-train")
SEEN_TRAIN = Counter()
DEV_BUCKETS = ["calc_easy", "calc_hard", "calc_overflow", "lookup_hit", "lookup_miss", "lookup_near_miss", "file_read", "unknown_fact",
               "distractor", "in_context_answerable", "undeclared_tool"]
DEV_PER_BUCKET = CFG["dev_per_bucket"]


def merge_dev_tables(dev):
    """The real gateway takes ONE --table per run: merge all dev worlds into one table, dropping any episode whose
    key/value conflicts or whose miss/near-miss key happens to exist in another world."""
    merged, keep = {}, []
    for e in dev:
        t = e["meta"].get("table")
        if t and any(k in merged and merged[k] != v for k, v in t.items()):
            continue
        if t:
            merged.update(t)
        keep.append(e)
    final = [e for e in keep if not (e["meta"].get("outputs") and e["meta"]["outputs"][0] in ("NONE", "NOT-FOUND") and e["meta"].get("key") in merged)]
    for e in final:
        if e["meta"].get("table") is not None:
            e["meta"]["table"] = merged            # one shared table, exactly what the Linux gateway will see
    return final, merged


def load_or_build_data():
    man_path = os.path.join(DATA_DIR, "manifest.json")
    if EXTERNAL_DATA and os.path.exists(os.path.join(EXTERNAL_DATA, "manifest.json")):
        man = json.load(open(os.path.join(EXTERNAL_DATA, "manifest.json")))
        for fn, h in man["files"].items():
            p = os.path.join(EXTERNAL_DATA, fn)
            assert os.path.exists(p), f"frozen dataset is missing {fn}"
            got = sha256_file(p)
            assert got == h, f"frozen dataset hash mismatch for {fn}: {got} != {h}"
            if os.path.abspath(EXTERNAL_DATA) != os.path.abspath(DATA_DIR):
                shutil.copy(p, os.path.join(DATA_DIR, fn))
        shutil.copy(os.path.join(EXTERNAL_DATA, "manifest.json"), man_path)
        print(f"frozen dataset verified: {len(man['files'])} files, SHA-256 match ({EXTERNAL_DATA})")
        return man, None, None
    # dev slice first (own rng), disjoint from training by construction + checked below
    drng = random.Random(f"{SEED}-dev")
    dev, dstats = [], Counter()
    for b in DEV_BUCKETS + ["mixed", "unanswerable"]:
        n = DEV_PER_BUCKET if b in DEV_BUCKETS else max(2, DEV_PER_BUCKET // 5)
        eps, st = build_episodes(drng, "dev", [b] * n)
        dev += eps
        dstats.update(st)
    dev, DEV_TABLE_ = merge_dev_tables(dev)
    n_ep = CFG["n_episodes"] or 1
    pilot, _ = build_episodes(random.Random("pilot"), "train", weighted_plan(random.Random("pilot-plan"), 300))
    avg_bytes = statistics.mean(len(episode_text(e)) for e in pilot)
    if CFG["n_episodes"] is None:
        n_ep = int(CFG["target_tokens"] * BYTES_PER_TOKEN_GUESS / avg_bytes * 1.05)
    plan = weighted_plan(TRAIN_RNG, n_ep)
    train, tstats = build_episodes(TRAIN_RNG, "train", plan, seen=SEEN_TRAIN)
    return None, (train, tstats, avg_bytes), (dev, dstats, DEV_TABLE_)


man, tr, dv = load_or_build_data()
if man is None:
    train_eps, tstats, avg_bytes = tr
    dev_eps, dstats, DEV_TABLE = dv
    # decontamination: dev prompts and canary must not occur in training; namespaces/prefixes/tables are disjoint by construction
    train_prompts = {e["segs"][0][1] for e in train_eps}
    dev_eps = [e for e in dev_eps if e["segs"][0][1] not in train_prompts]
    assert not any(CANARY in episode_text(e) for e in train_eps), "canary leaked into training data"
    tr_rate = 1 - tstats["rejected"] / max(1, tstats["generated"])
    print(f"train episodes kept {len(train_eps)} of {tstats['generated']} generator draws; round-trip survival {tr_rate:.1%}; dev episodes {len(dev_eps)}")
    assert tr_rate >= .80, "S1 kill gate: under 80% of generated episodes survive the round trip, so the generator is wrong"
    assert len(dev_eps) > 0
    rej = {k: v for k, v in tstats.items() if k.startswith("rejected:")}
    sat = {k: v for k, v in tstats.items() if k.startswith("skipped_saturated")}
    print("round-trip rejections:", rej or "none", "| repeats suppressed:", tstats["duplicate"], "| saturated pools:", sat or "none")
    print("train mix:", dict(Counter(e["generator"] for e in train_eps).most_common()))
    print("dev mix  :", dict(Counter(e["bucket"] for e in dev_eps).most_common()))
    print(f"mean episode length {avg_bytes:.0f} bytes (pilot)")
    for g in ("calc_easy", "calc_overflow", "unknown_fact", "mixed", "undeclared_tool"):
        e = next(e for e in train_eps if e["generator"] == g)
        print(f"--- sample [{g}] ---\n" + episode_text(e).replace("\n", "\\n\n"), end="")
        print()
else:
    train_eps = dev_eps = None
    DEV_TABLE = {k: v for k, v in (l.rstrip("\n").split("\t", 1) for l in open(os.path.join(DATA_DIR, "dev_table.tsv")) if l.strip())}


def factory_source_sha256():
    try:
        import inspect
        fns = [calc_oracle, find_calc, find_tool_call, run_tool, roundtrip, build_episodes, weighted_plan] + [f for f, _ in GENERATORS.values()]
        return hashlib.sha256("".join(inspect.getsource(f) for f in fns).encode()).hexdigest()
    except Exception as e:  # noqa
        return "unavailable:" + type(e).__name__


def freeze_manifest(train_eps, dev_eps, extra=None):
    paths = {"episodes.train.jsonl": os.path.join(DATA_DIR, "episodes.train.jsonl"), "episodes.dev.jsonl": os.path.join(DATA_DIR, "episodes.dev.jsonl"),
             "dev_table.tsv": os.path.join(DATA_DIR, "dev_table.tsv")}
    write_jsonl(paths["episodes.train.jsonl"], train_eps, False, "train")
    write_jsonl(paths["episodes.dev.jsonl"], dev_eps, True, "dev")
    with open(paths["dev_table.tsv"], "w") as f:       # key<TAB>value, the format agent_trace --table parses (sorted: deterministic bytes)
        for k in sorted(DEV_TABLE):
            f.write(f"{k}\t{DEV_TABLE[k]}\n")
    man = {"dataset": "e1-evolve-pilot-episodes", "factory_version": GEN_VERSION, "seed": SEED, "smoke": SMOKE,
           "format": "agent_trace raw Q:/A: text; segs = [kind, text], kind in ctx|gen|tool (loss only on gen)",
           "n_train_episodes": len(train_eps), "n_dev_episodes": len(dev_eps),
           "generators": dict(Counter(e["generator"] for e in train_eps)), "dev_buckets": dict(Counter(e["bucket"] for e in dev_eps)),
           "canary": CANARY, "factory_source_sha256": factory_source_sha256(), "files": {}}
    for fn, p in paths.items():
        man["files"][fn] = sha256_file(p)
    if extra:
        man.update(extra)
    json.dump(man, open(os.path.join(DATA_DIR, "manifest.json"), "w"), indent=1, sort_keys=True)
    return man


if man is None:
    man = freeze_manifest(train_eps, dev_eps, {"status": "preliminary (token counts are added by Cell 2)"})
    print("manifest written:", json.dumps(man["files"], indent=1))

train episodes kept 1200 of 1201 generator draws; round-trip survival 100.0%; dev episodes 48
round-trip rejections: none | repeats suppressed: 1 | saturated pools: none
train mix: {'unknown_fact': 141, 'calc_hard': 134, 'undeclared_tool': 125, 'distractor': 106, 'calc_easy': 97, 'mixed': 95, 'unanswerable_open': 90, 'lookup_hit': 87, 'calc_overflow': 80, 'file_read': 78, 'in_context_answerable': 73, 'lookup_miss': 48, 'lookup_near_miss': 46}
dev mix  : {'calc_easy': 4, 'calc_hard': 4, 'calc_overflow': 4, 'lookup_hit': 4, 'lookup_miss': 4, 'lookup_near_miss': 4, 'file_read': 4, 'unknown_fact': 4, 'distractor': 4, 'in_context_answerable': 4, 'undeclared_tool': 4, 'mixed': 2, 'unanswerable': 2}
mean episode length 135 bytes (pilot)
--- sample [calc_easy] ---
[tools] CALC, FILE-READ:docs16\n
Q: Work out 16 + 13\n
A: CALC(16 + 13).\n
TOOL[calc]=29\n
A: That is 29.\n

--- sample [calc_overflow] ---
You may call: CALC.\n
Q: Calculate -9223372036854775808 % -1\n
A: CALC(-9223372036854775808 %

## Cell 2 - tokenizer parity (G1): this notebook PORTS the engine's `encode()`

**Which of the two options this notebook takes: the port.** All training and evaluation text is tokenized by
`EngineTokenizer.encode`, a line-by-line Python port of `AegisTokenizer::encode` (`aegis-core/src/tokenizer.rs`) over the
**pruned 50,256-id vocabulary** (Llama-3 base ids `< 50000` unchanged; the 256 added specials remapped `128000+k -> 50000+k`,
exactly `repack_ternary.py --llama3-prune`). The engine splits only at whitespace/non-whitespace transitions (a trailing space of
a whitespace run detaches to prefix the next word), then runs byte-level BPE inside each piece; it is **not** the Llama-3 regex.

What is checked here, and what is not:
* the vocabulary and merge table are rebuilt exactly as `write_vocab_bin` does and the SHA-256 of the resulting `VOCAB.BIN` bytes is compared with the
  pinned incumbent `VOCAB.BIN` (`5bde1b03...`, LAB-07 s2) -> proves the port encodes against the engine's table;
* round trip `decode(encode(x)) == x` on every training/eval text, no byte falls back to `<|reserved_special_token_0|>`;
* the **known skew cases** of design 01 s2.6 are tokenized by both the port and the HF Llama-3 tokenizer and printed side by side
  (they are expected to DIFFER; this is the reason the port exists);
* **not checked here:** 100% id parity against the *Rust* `encode()` on >= 10 MB (gate G1). That needs the Rust binary and runs on the Linux side:
  this cell writes `g1_vectors.jsonl` (text + ids + SHA-256) and the README gives the command. Until it passes, G1 is "port written, Rust parity pending".

In [5]:
import numpy as np

PINNED_VOCAB_SHA256 = "5bde1b0355ef99c6875190ebfff081d985ca48977ae9269e3477f5cc2d97d9ae"   # incumbent VOCAB.BIN (LAB-07 s2, LAB-02 repack log)
VOCAB_MAGIC = 0x564F4341


def byte_to_unicode(b):
    if 33 <= b <= 126 or 161 <= b <= 172 or 174 <= b <= 255:
        return b
    if 0 <= b <= 32:
        return 256 + b
    if 127 <= b <= 160:
        return 256 + 33 + (b - 127)
    return 323      # b == 173


BYTE2CH = [chr(byte_to_unicode(b)) for b in range(256)]
CH2BYTE = {c: b for b, c in enumerate(BYTE2CH)}
_WS_ASCII = " \t\n\x0b\x0c\r"
_RUNS = re.compile(r"[ \t\n\x0b\x0c\r]+|[^ \t\n\x0b\x0c\r]+")


class EngineTokenizer:
    """Python port of aegis-core AegisTokenizer::encode/decode over the repack_ternary --llama3-prune id space."""

    def __init__(self, tok_json, cut=50000, special_base=128000):
        base = tok_json["model"]["vocab"]
        kept = sorted((oid, s) for s, oid in base.items() if oid < cut)
        assert len(kept) == cut and all(oid == i for i, (oid, _) in enumerate(kept)), "base ids below the cut must be dense"
        specials = sorted((t["id"], t["content"]) for t in tok_json.get("added_tokens", []))
        assert all(i >= special_base for i, _ in specials), "special ids must sit at/above special_base"
        self.cut, self.special_base, self.n_special = cut, special_base, len(specials)
        self.tokens = [s for _, s in kept] + [s for _, s in specials]
        self.old_ids = [i for i, _ in kept] + [i for i, _ in specials]
        self.vocab_size = len(self.tokens)
        self.str_to_id = {s: i for i, s in enumerate(self.tokens)}
        assert len(self.str_to_id) == len(self.tokens), "duplicate token strings"
        self.kept_merges = []
        for m in tok_json["model"].get("merges", []):
            p1, p2 = (m.split(" ") if isinstance(m, str) else m)
            i1, i2, im = self.str_to_id.get(p1), self.str_to_id.get(p2), self.str_to_id.get(p1 + p2)
            if i1 is not None and i2 is not None and im is not None:
                self.kept_merges.append((i1, i2, im))
        self.merges = {}
        for rank, (a, b, m) in enumerate(self.kept_merges):
            if a != 0 or b != 0 or m != 0:
                self.merges[(a, b)] = (m, rank)          # BTreeMap insert: the last duplicate wins
        self.unk = self.str_to_id.get("<|reserved_special_token_0|>")
        self.char_id = [self.str_to_id.get(c) for c in BYTE2CH]
        self.cache = {}
        self.eos = self.str_to_id.get("<|end_of_text|>")
        self.bos = self.str_to_id.get("<|begin_of_text|>")
        # pruned id -> original (HF) id, and the list of original ids the head ranks over
        self.to_orig = [oid for oid in self.old_ids]

    # -- VOCAB.BIN bytes exactly as repack_ternary.write_vocab_bin
    def vocab_bin_bytes(self):
        b = bytearray(struct.pack("<II", VOCAB_MAGIC, len(self.tokens)))
        for s in self.tokens:
            e = s.encode("utf-8")
            b += struct.pack("<H", len(e)) + e
        b += struct.pack("<I", len(self.kept_merges))
        for tri in self.kept_merges:
            b += struct.pack("<III", *tri)
        return bytes(b)

    def _merge_word(self, w):
        w = list(w)
        while len(w) >= 2:
            best_rank, best_idx, best_m = 1 << 62, -1, None
            for j in range(len(w) - 1):
                e = self.merges.get((w[j], w[j + 1]))
                if e is not None and e[1] < best_rank:
                    best_rank, best_idx, best_m = e[1], j, e[0]
            if best_m is None:
                break
            w[best_idx] = best_m
            del w[best_idx + 1]
        return w

    def _piece_ids(self, piece):
        r = self.cache.get(piece)
        if r is None:
            init = []
            for b in piece.encode("latin-1"):
                i = self.char_id[b]
                if i is None:
                    i = self.unk
                if i is not None:
                    init.append(i)
            r = tuple(self._merge_word(init))
            self.cache[piece] = r
        return r

    @staticmethod
    def pieces(text):
        """The engine's pre-tokenizer: split at whitespace<->non-whitespace transitions; a whitespace run that ends in a plain
        space and is followed by a word gives that space to the word (byte-level `Ġ?\\S+` semantics)."""
        out, pending = [], ""
        n = len(text)
        for m in _RUNS.finditer(text):
            run = m.group()
            if run[0] in _WS_ASCII:
                if m.end() < n and run[-1] == " ":
                    if len(run) > 1:
                        out.append(run[:-1])
                    pending = " "
                else:
                    out.append(run)
                    pending = ""
            else:
                out.append(pending + run)
                pending = ""
        return out

    def encode(self, text):
        assert text.isascii(), "the engine path is ASCII-only; filter non-ASCII before tokenizing"
        ids = []
        for p in self.pieces(text):
            ids.extend(self._piece_ids(p))
        return ids

    def encode_offsets(self, text):
        """ids plus the byte offset of each token's end (for mapping loss spans to tokens)."""
        ids, ends, pos = [], [], 0
        for p in self.pieces(text):
            for i in self._piece_ids(p):
                s = self.tokens[i]
                assert len(s) <= len(p), "unmapped byte fell back to the reserved token"
                pos += len(s)
                ids.append(i)
                ends.append(pos)
        assert pos == len(text), "offsets do not cover the text"
        return ids, ends

    def decode(self, ids):
        buf = bytearray()
        for i in ids:
            if 0 <= i < len(self.tokens):
                for ch in self.tokens[i]:
                    u = ord(ch)
                    if u < 256:
                        buf.append(u)
                    else:
                        off = u - 256
                        buf.append(off if off <= 32 else (off - 33 + 127) if 33 <= off <= 66 else 173 if off == 67 else ord("?"))
        return buf.decode("utf-8", errors="replace")


def load_tokenizer_json():
    """Real run: the tokenizer.json of the base repo (plus tokenizer_config.json for the export). Smoke without network: a tiny trained fallback."""
    from huggingface_hub import hf_hub_download
    if os.environ.get("E1_OFFLINE") == "1":
        return None, None
    try:
        tj = hf_hub_download(BASE_REPO, "tokenizer.json")
        try:
            tc = hf_hub_download(BASE_REPO, "tokenizer_config.json")
        except Exception:
            tc = None
        return tj, tc
    except Exception as e:  # noqa
        print("tokenizer download failed:", type(e).__name__, str(e)[:120])
        return None, None


def train_fallback_tokenizer(texts, out_dir):
    """Offline smoke only: a tiny byte-level BPE trained on the generated corpus, laid out like the Llama-3 id space
    (base ids 0..N-1, 8 specials appended) so every code path of the prune/remap logic still runs."""
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
    tk.decoder = decoders.ByteLevel()
    tr = trainers.BpeTrainer(vocab_size=1100, special_tokens=[], initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tk.train_from_iterator(texts, tr)
    path = os.path.join(out_dir, "tokenizer_fallback.json")
    tk.save(path)
    j = json.load(open(path))
    n = len(j["model"]["vocab"])
    names = ["<|begin_of_text|>", "<|end_of_text|>"] + [f"<|reserved_special_token_{i}|>" for i in range(6)]
    j["added_tokens"] = [{"id": n + k, "content": s, "special": True} for k, s in enumerate(names)]
    json.dump(j, open(path, "w"))
    return path, n


TOK_JSON, TOK_CFG = load_tokenizer_json()
if TOK_JSON is None:
    assert SMOKE, "tokenizer.json could not be fetched; a real run needs the base repo reachable (or attach it as a dataset)"
    texts = [episode_text(e) for e in train_eps[:1500]]
    TOK_JSON, n_base = train_fallback_tokenizer(texts, WORK)
    TOK_CUT, TOK_SPECIAL_BASE = int(n_base * 0.6), n_base
    FALLBACK_TOKENIZER = True
    print(f"[smoke] fallback tokenizer: {n_base} base ids, cut {TOK_CUT}, specials at {n_base}")
else:
    TOK_CUT, TOK_SPECIAL_BASE = 50000, 128000
    FALLBACK_TOKENIZER = False
tok_j = json.load(open(TOK_JSON))
ET = EngineTokenizer(tok_j, cut=TOK_CUT, special_base=TOK_SPECIAL_BASE)
FULL_VOCAB = TOK_SPECIAL_BASE + ET.n_special          # embedding rows of the base checkpoint (128256 for the real model)
KEPT_ORIG = np.array(ET.old_ids, dtype=np.int64)       # pruned id -> HF id; the head ranks over these rows only
print(f"engine-port tokenizer: pruned vocab {ET.vocab_size} ids, {len(ET.kept_merges)} merges, full table {FULL_VOCAB} rows, eos(pruned)={ET.eos}")

vb = ET.vocab_bin_bytes()
vb_sha = hashlib.sha256(vb).hexdigest()
if not FALLBACK_TOKENIZER:
    status = "PASS" if vb_sha == PINNED_VOCAB_SHA256 else "FAIL"
    print(f"VOCAB.BIN rebuilt from tokenizer.json: {len(vb)} bytes sha256 {vb_sha}\n  pinned incumbent VOCAB.BIN sha256       {PINNED_VOCAB_SHA256}  -> {status}")
    assert status == "PASS", "the port's vocabulary/merge table is NOT the engine's: stop before training (G1)"
else:
    print(f"[smoke] fallback VOCAB.BIN-equivalent sha256 {vb_sha[:16]}... (no pinned value for a toy tokenizer)")

# ---- known skew cases of design 01 s2.6: engine-port vs the HF Llama-3 tokenizer
from tokenizers import Tokenizer as HFTokenizer
hf_tok = HFTokenizer.from_file(TOK_JSON)
SKEW_CASES = ["9223372036854775807", "CALC(9223372036854775807 + 1).\n", "Q: part P-206\nA: LOOKUP(P-206).\n", "x).\ny"]


def show(ids):
    return "|".join(ET.decode([i]).replace("\n", "\\n") if i < ET.vocab_size else f"<{i}>" for i in ids)


skew_report = []
for t in SKEW_CASES:
    a = ET.encode(t)
    h = hf_tok.encode(t, add_special_tokens=False).ids
    h_pruned = [i for i in h if i < TOK_CUT]
    differ = (a != h)
    skew_report.append({"text": t, "engine_port": show(a), "hf_llama3": "|".join(hf_tok.decode([i]).replace("\n", "\\n") for i in h), "differ": differ})
    print(f"{t!r}\n   engine-port ({len(a)}): {show(a)}\n   HF llama-3  ({len(h)}): {skew_report[-1]['hf_llama3']}   -> {'DIFFER' if differ else 'same'}")
if not FALLBACK_TOKENIZER:
    # design 01 s2.6 / K4: the two documented skews. They must reproduce on the real vocabulary, otherwise the premise of the port changed.
    assert skew_report[0]["differ"], "expected the 19-digit number to tokenize differently (design 01 s2.6)"
    assert skew_report[1]["differ"] or skew_report[3]["differ"], "expected the ').\\n' site to tokenize differently (design 01 s2.6)"

# ---- round trip on random ASCII and on every template family
_rt = random.Random(5)
alphabet = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 \n\t()[]=:.,-_/+*%?!'\"#"
for _ in range(2000):
    s = "".join(_rt.choice(alphabet) for _ in range(_rt.randrange(0, 60)))
    ids = ET.encode(s)
    assert ET.decode(ids) == s, repr(s)
print("round trip decode(encode(x)) == x on 2000 random ASCII strings: OK")

engine-port tokenizer: pruned vocab 50256 ids, 110042 merges, full table 128256 rows, eos(pruned)=50001
VOCAB.BIN rebuilt from tokenizer.json: 1759936 bytes sha256 5bde1b0355ef99c6875190ebfff081d985ca48977ae9269e3477f5cc2d97d9ae
  pinned incumbent VOCAB.BIN sha256       5bde1b0355ef99c6875190ebfff081d985ca48977ae9269e3477f5cc2d97d9ae  -> PASS


'9223372036854775807'
   engine-port (8): 922|33|720|36|85|47|75|807
   HF llama-3  (7): 922|337|203|685|477|580|7   -> DIFFER
'CALC(9223372036854775807 + 1).\n'
   engine-port (17): C|AL|C|(|922|33|720|36|85|47|75|807| +| |1|).|\n
   HF llama-3  (14): CAL|C|(|922|337|203|685|477|580|7| +| |1|).\n   -> DIFFER
'Q: part P-206\nA: LOOKUP(P-206).\n'
   engine-port (17): Q|:| part| P|-|206|\n|A|:| LO|OK|UP|(P|-|206|).|\n
   HF llama-3  (15): Q|:| part| P|-|206|\n|A|:| LOOK|UP|(P|-|206|).\n   -> DIFFER
'x).\ny'
   engine-port (4): x|).|\n|y
   HF llama-3  (3): x|).\n|y   -> DIFFER
round trip decode(encode(x)) == x on 2000 random ASCII strings: OK


In [6]:
# ---------------------------------------------------------------- tokenize the frozen episodes through the engine port, trim/extend to the token budget, pack
SEQ = CFG["seq"]
PAD_ID = ET.eos


def tokenize_episode(ep):
    """ids + loss mask for one episode (loss only on 'gen' segments + the closing EOS)."""
    text = episode_text(ep)
    ids, ends = ET.encode_offsets(text)
    spans, pos = [], 0
    for kind, t in ep["segs"]:
        spans.append((pos, pos + len(t), kind))
        pos += len(t)
    mask, si, start = [], 0, 0
    for end in ends:
        while spans[si][1] <= start:
            si += 1
        a, b, kind = spans[si]
        assert end <= b, "token straddles a segment boundary: the loss mask would be ambiguous"
        mask.append(1 if kind == "gen" else 0)
        start = end
    return ids + [ET.eos], mask + [1]


def pack_sequences(tokenized, seq_len):
    """Greedy whole-episode packing into rows of seq_len+1 (input = row[:-1], target = row[1:]); the tail of a row is EOS padding with loss off."""
    rows_i, rows_m, cur_i, cur_m, real = [], [], [], [], 0
    for ids, m in tokenized:
        assert len(ids) <= seq_len + 1, "an episode is longer than the sequence length"
        if len(cur_i) + len(ids) > seq_len + 1:
            rows_i.append(cur_i + [PAD_ID] * (seq_len + 1 - len(cur_i)))
            rows_m.append(cur_m + [0] * (seq_len + 1 - len(cur_m)))
            cur_i, cur_m = [], []
        cur_i += ids
        cur_m += m
        real += len(ids)
    if cur_i:
        rows_i.append(cur_i + [PAD_ID] * (seq_len + 1 - len(cur_i)))
        rows_m.append(cur_m + [0] * (seq_len + 1 - len(cur_m)))
    return np.asarray(rows_i, dtype=np.int32), np.asarray(rows_m, dtype=np.uint8), real


def finalize_dataset():
    global train_eps
    arr_ids, arr_mask = os.path.join(DATA_DIR, "train_seqs.npy"), os.path.join(DATA_DIR, "train_mask.npy")
    if train_eps is None:       # frozen external dataset: arrays and dev file come from the verified manifest
        man2 = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
        assert man2.get("seq_len") == SEQ, f"frozen dataset was packed at seq_len {man2.get('seq_len')}, this run uses {SEQ}"
        devs = [json.loads(l) for l in open(os.path.join(DATA_DIR, "episodes.dev.jsonl"))]
        return np.load(arr_ids), np.load(arr_mask), devs, man2, man2.get("real_tokens", 0)
    target = CFG["target_tokens"]
    tokenized, total = [], 0
    kept = []
    i = 0
    while True:
        if i >= len(train_eps):
            if not target or total >= target:
                break
            k = max(1000, int((target - total) / max(1.0, total / max(1, len(kept))) * 1.1))
            more, _ = build_episodes(TRAIN_RNG, "train", weighted_plan(TRAIN_RNG, k), seen=SEEN_TRAIN)
            train_eps += more
            print(f"  extended the corpus by {len(more)} episodes to reach the token budget")
        ep = train_eps[i]
        t = tokenize_episode(ep)
        tokenized.append(t)
        kept.append(ep)
        total += len(t[0])
        i += 1
        if target and total >= target:
            break
    train_eps = kept
    X, M, real = pack_sequences(tokenized, SEQ)
    assert real == total
    np.save(arr_ids, X)
    np.save(arr_mask, M)
    return X, M, [json.loads(l) for l in open(os.path.join(DATA_DIR, "episodes.dev.jsonl"))], None, total


TRAIN_X, TRAIN_M, DEV_RAW, man_ext, real_tokens = finalize_dataset()
if man_ext is None:
    n_bytes = int(sum(len(episode_text(e)) for e in train_eps))
    tokens_total = int(TRAIN_M.size)                                           # packed positions (incl. EOS padding)
    n_eps = len(train_eps)
    loss_tokens = int(TRAIN_M.sum())
    print(f"train: {n_eps} episodes -> {TRAIN_X.shape[0]} rows x {SEQ + 1} = {tokens_total} packed positions; "
          f"{real_tokens / 1e6:.2f}M real tokens; loss on {loss_tokens} tokens ({loss_tokens / tokens_total:.1%} of positions)")
    print(f"bytes per token (engine port): {n_bytes / max(1, real_tokens):.2f}")
    if not SMOKE and not (10e6 <= real_tokens <= 20e6 * 1.05):
        print(f"WARNING: token budget {real_tokens / 1e6:.2f}M is outside the 10-20M target of the plan")
    # G10 manifest: freeze (re-write when the corpus was trimmed or extended), then add token statistics
    man = freeze_manifest(train_eps, [dict(e) for e in dev_eps], {
        "status": "frozen", "seq_len": SEQ, "tokenizer": {"kind": "engine-port", "vocab_bin_rebuilt_sha256": vb_sha,
                                                          "pinned_incumbent_vocab_sha256": PINNED_VOCAB_SHA256,
                                                          "fallback_toy_tokenizer": FALLBACK_TOKENIZER},
        "rows": int(TRAIN_X.shape[0]), "packed_positions": tokens_total, "real_tokens": real_tokens, "loss_tokens": loss_tokens})
    man["files"]["train_seqs.npy"] = sha256_file(os.path.join(DATA_DIR, "train_seqs.npy"))
    man["files"]["train_mask.npy"] = sha256_file(os.path.join(DATA_DIR, "train_mask.npy"))
    json.dump(man, open(os.path.join(DATA_DIR, "manifest.json"), "w"), indent=1, sort_keys=True)
    print("frozen manifest sha256:", sha256_file(os.path.join(DATA_DIR, "manifest.json")))

    # G1 vectors for the Linux-side Rust comparison (text as hex + the ids this notebook trains on)
    nvec = min(len(train_eps), CFG["g1_vectors"])
    with open(os.path.join(WORK, "g1_vectors.jsonl"), "w") as f:
        for e in train_eps[:nvec] + [{"segs": [["ctx", t]]} for t in SKEW_CASES]:
            t = episode_text(e)
            f.write(json.dumps({"text_hex": t.encode().hex(), "ids": ET.encode(t)}, separators=(",", ":")) + "\n")
    print(f"g1_vectors.jsonl: {nvec + len(SKEW_CASES)} texts, sha256 {sha256_file(os.path.join(WORK, 'g1_vectors.jsonl'))[:16]}...")
    open(os.path.join(WORK, "VOCAB.BIN.rebuilt"), "wb").write(vb)

# the token-stream sanity: every loss token is a model-turn token and the first loss token of an episode is a space-prefixed word
_ids = TRAIN_X[0]
_m = TRAIN_M[0]
_first = int(np.argmax(_m))
print("row 0 decoded (loss tokens in [[...]]):")
print(("".join(("[[" + ET.decode([int(t)]) + "]]") if mm else ET.decode([int(t)]) for t, mm in zip(_ids[:160], _m[:160]))).replace("\n", "\\n"))

train: 1200 episodes -> 679 rows x 129 = 87591 packed positions; 0.07M real tokens; loss on 25731 tokens (29.4% of positions)
bytes per token (engine port): 2.42


frozen manifest sha256: a8f15115401cacdd99afc7ddcb594e92962d4ced8125dfea003c13f0388c5401
g1_vectors.jsonl: 204 texts, sha256 828fd22c1efad977...
row 0 decoded (loss tokens in [[...]]):
TOOLS: CALC, LOOKUP:assets\nQ: What does the record say about UW5373?\nA:[[ LO]][[OK]][[UP]][[(U]][[W]][[5]][[373]][[).]]\nTOOL[lookup]=NONE\n[[A]][[:]][[ U]][[W]][[5]][[373]][[ is]][[ not]][[ in]][[ the]][[ table]][[.]][[\n]][[<|end_of_text|>]]TOOLS: LOOKUP:orders\nQ: Find ZB-Y967 in the orders table.\nA:[[ LO]][[OK]][[UP]][[(Z]][[B]][[-Y]][[967]][[).]]\nTOOL[lookup]=17637\n[[A]][[:]][[ The]][[ table]][[ lists]][[ Z]][[B]][[-Y]][[967]][[ as]][[ ]][[176]][[37]][[\n]][[<|end_of_text|>]]<|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|><|end_of_text|>


## Cell 3 - base reference, three training arms, kill gates

**Baseline in the same harness.** Before any update, the *unmodified incumbent* is run through exactly the same held-out text, dev slice and scorer (that is the arm's "step 0").
Two stock references, both independent of this notebook's QAT wrapper (they use Hugging Face's own online `AutoBitLinear`), both restricted to the 50,256 ids the engine serves:
the **gate reference** keeps fp32 master weights under the *same autocast* as the training forward, so the only intended difference is the int8 head-input quantization (CIS point 5, which the stock forward lacks);
the **informational reference** is the checkpoint's native bf16 forward, whose rounding noise flips int8 activation roundings and moves argmax on its own. Both are printed at step 0, plus an attribution line with the head quantization switched off.

**Gates printed at every eval** (all are GATES, not predictions):
* *step-0 parity* - fraction of next-token argmax disagreements vs the reference on held-out text, **<= 3%**. It is a gate at step 0 (model == base); at later evals the same number is printed as drift, informational.
* *perplexity ratio* - model PPL / reference PPL on the fixed held-out slice, **<= 1.05** (flag above 1.02). Above 1.05 the arm is stopped (S3/S4 kill gate).
* *P0 `unknown_fact` proxy* - fraction of held-out `unknown_fact` dev items whose **greedy continuation begins with `LOOKUP(<exact key>)`** with **zero demonstration in the prompt** (the prompt is a one-line tool declaration, no example call), **>= 73.3%**
  (the incumbent's measured T2 value, 22/30, LAB-07 s8). `distractor` no-tool precision is printed beside it and must not fall below the step-0 value of the same arm.
* T2 (two `LOOKUP` shots on other keys) is evaluated at step 0 and at the end as a diagnostic that continues LAB-07.

The dev slice is built from **held-out namespaces, phrasings and header families**; it is a movement test (wide Wilson intervals), not a ship test. Scoring here is a *proxy* of `score.py`: the real receipts come from the Linux gateway.

In [7]:
import contextlib

KEPT_IDX = torch.as_tensor(KEPT_ORIG, dtype=torch.long)          # pruned id -> HF id
EOS_ORIG = int(KEPT_ORIG[ET.eos])
if SMOKE and FULL_VOCAB != SMOKE_VOCAB:                           # offline smoke with the toy tokenizer: rebuild the tiny base at the toy vocabulary
    BASE_SRC = make_tiny_base(FULL_VOCAB, os.path.join(WORK, "tiny_base"))
    print("[smoke] tiny base rebuilt at vocab", FULL_VOCAB)


def autocast_ctx(enabled=True):
    if not enabled:
        return contextlib.nullcontext()
    return torch.autocast("cuda", dtype=AMP_DTYPE) if CUDA else torch.autocast("cpu", dtype=torch.bfloat16)


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


# ------------------------------------------------------------------ held-out text (ASCII, engine-port ids)
def synthetic_prose(rng, n_chars):
    subj = ["The operator", "A technician", "The crew", "Our supervisor", "The night shift", "A visitor", "The driver", "Two engineers"]
    verb = ["checked", "inspected", "moved", "repaired", "logged", "cleaned", "weighed", "labelled"]
    obj = ["the pump", "a spare valve", "the loading dock", "two crates", "the old filter", "the east gate", "every pallet", "the cold room"]
    tail = ["before lunch.", "after the alarm.", "without delay.", "as planned.", "twice that week.", "and wrote it down.", "on the third try."]
    out, n = [], 0
    while n < n_chars:
        s = f"{rng.choice(subj)} {rng.choice(verb)} {rng.choice(obj)} {rng.choice(tail)}"
        out.append(s)
        n += len(s) + 1
    return " ".join(out)


def build_heldout():
    need = CFG["ppl_seqs"] * CFG["ppl_len"]
    text, src = None, None
    if not SMOKE or os.environ.get("E1_WIKITEXT") == "1":
        try:
            from datasets import load_dataset
            ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
            text = "\n".join(ds["text"]).encode("ascii", "ignore").decode()
            src = "wikitext-2-raw-v1 test (ASCII-filtered; the program's PPL anchor; evaluation use only)"
        except Exception as e:  # noqa
            print("WikiText-2 unavailable (", type(e).__name__, ") -> synthetic prose fallback; PPL is then NOT comparable with the program anchor")
    if text is None:
        text = synthetic_prose(random.Random(99), need * 6)
        src = "synthetic prose (fallback; not comparable with the program anchor)"
    ids = np.asarray(ET.encode(text), dtype=np.int64)
    n_win = len(ids) // CFG["ppl_len"]
    assert n_win >= CFG["ppl_seqs"], f"held-out text yields only {n_win} windows of {CFG['ppl_len']} tokens"
    return ids[: CFG["ppl_seqs"] * CFG["ppl_len"]].reshape(CFG["ppl_seqs"], CFG["ppl_len"]), src


HELD, HELD_SRC = build_heldout()
HELD_SHA = hashlib.sha256(HELD.tobytes()).hexdigest()
print(f"held-out slice: {HELD.shape[0]} windows x {HELD.shape[1]} engine-port tokens, source: {HELD_SRC}, sha256 {HELD_SHA[:16]}...")


# ------------------------------------------------------------------ dev slice from the held-out episodes
def build_dev_items(raw):
    items = []
    for e in raw:
        meta = e["meta"]
        ctx = e["segs"][0][1]
        items.append({"bucket": e["bucket"], "generator": e["generator"], "ns": e["ns"], "prompt": ctx, "calls": meta.get("calls", []),
                      "outputs": meta.get("outputs", []), "key": meta.get("key"), "table_present": bool(meta.get("table"))})
    return items


DEV_ITEMS = build_dev_items(DEV_RAW)
NO_CALL_BUCKETS = {"distractor", "in_context_answerable", "undeclared_tool", "unanswerable"}
SCORED_BUCKETS = [b for b in DEV_BUCKETS] + ["unanswerable"]


def t2_prompt(item, rng):
    """T2 of LAB-07 s8: two LOOKUP shots on OTHER keys, same question; header first."""
    head, rest = item["prompt"].split("\n", 1)
    ns = NS_DEV[item["ns"]] if item["ns"] >= 0 else NS_DEV[0]
    shots = []
    while len(shots) < 2:
        k = ns.key(rng)
        if k != item["key"]:
            shots.append(f"Q: {ns.entity} {k}\nA: LOOKUP({k}).\n")
    return head + "\n" + "".join(shots) + rest


def score_item(it, text):
    """Proxy of score.py: call buckets need the FIRST scanned call to equal the expected call verbatim (a LOOKUP with an extra argument fails);
    no-call buckets need no call opener anywhere in the generated text."""
    if it["bucket"] in NO_CALL_BUCKETS:
        return not has_call_opener(text)
    r = find_tool_call(text, True)
    return r is not None and r[1] == it["calls"][0]


# ------------------------------------------------------------------ greedy generation with a KV cache over the served ids (engine decodes exactly N tokens; EOS ignored)
@torch.no_grad()
def greedy_generate(model, prompts_pruned, max_new, bs, amp=True):
    was_training = model.training
    model.eval()
    dev = embed_weight(model).device
    order = sorted(range(len(prompts_pruned)), key=lambda i: len(prompts_pruned[i]))
    outs = [None] * len(prompts_pruned)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        P = [prompts_pruned[i] for i in idx]
        L = max(len(p) for p in P)
        ids = torch.full((len(P), L), EOS_ORIG, dtype=torch.long)
        att = torch.zeros((len(P), L), dtype=torch.long)
        for r_, p in enumerate(P):
            ids[r_, L - len(p):] = KEPT_IDX[torch.as_tensor(p, dtype=torch.long)]
            att[r_, L - len(p):] = 1
        ids, att = ids.to(dev), att.to(dev)
        pos = (att.cumsum(-1) - 1).clamp(min=0)
        gen = []
        with autocast_ctx(amp):
            o = backbone(model)(input_ids=ids, attention_mask=att, position_ids=pos, use_cache=True)
            cache = o.past_key_values
            h = o.last_hidden_state[:, -1:]
            nxt = head_logits(model, h, KEPT_IDX)[:, 0].argmax(-1)
            gen.append(nxt)
            last_pos = pos[:, -1:]
            for _ in range(max_new - 1):
                att = torch.cat([att, torch.ones((att.shape[0], 1), dtype=att.dtype, device=att.device)], 1)
                last_pos = last_pos + 1
                o = backbone(model)(input_ids=KEPT_IDX[nxt.cpu()][:, None].to(dev), attention_mask=att, position_ids=last_pos, past_key_values=cache, use_cache=True)
                cache = o.past_key_values
                nxt = head_logits(model, o.last_hidden_state[:, -1:], KEPT_IDX)[:, 0].argmax(-1)
                gen.append(nxt)
        G = torch.stack(gen, 1).cpu().tolist()
        for r_, i in enumerate(idx):
            outs[i] = G[r_]
    if was_training:
        model.train()
    return outs


def generation_selfcheck(model, n=6):
    """Batched left-padded generation must agree with one-at-a-time generation (guards the cache / mask / position-id plumbing)."""
    ps = [ET.encode(it["prompt"]) for it in DEV_ITEMS[:: max(1, len(DEV_ITEMS) // n)][:n]]
    b = greedy_generate(model, ps, 8, bs=n, amp=False)         # fp32: removes the autocast rounding noise from the comparison
    s = [greedy_generate(model, [p], 8, bs=1, amp=False)[0] for p in ps]
    same = sum(int(x == y) for u, v in zip(b, s) for x, y in zip(u, v)) / (8 * len(ps))
    return same


def eval_dev(model, with_t2=False):
    prompts = [ET.encode(it["prompt"]) for it in DEV_ITEMS]
    texts = [ET.decode(g) for g in greedy_generate(model, prompts, CFG["gen_new"], CFG["gen_bs"])]
    res = {"P0": defaultdict(lambda: [0, 0])}
    strict = [0, 0]
    for it, t in zip(DEV_ITEMS, texts):
        if it["bucket"] == "mixed":
            continue
        ok = score_item(it, t)
        res["P0"][it["bucket"]][0] += int(ok)
        res["P0"][it["bucket"]][1] += 1
        if it["bucket"] == "unknown_fact":
            strict[1] += 1
            strict[0] += int(t.lstrip().startswith(it["calls"][0]))
    out = {"P0": {b: {"k": v[0], "n": v[1], "rate": v[0] / max(1, v[1]), "wilson": wilson(*v)} for b, v in res["P0"].items()},
           "p0_unknown_strict": {"k": strict[0], "n": strict[1], "rate": strict[0] / max(1, strict[1]), "wilson": wilson(*strict)},
           "sample_continuations": {it["bucket"]: t for it, t in zip(DEV_ITEMS, texts) if it["bucket"] in ("unknown_fact", "calc_hard", "distractor")}}
    if with_t2:
        rng = random.Random(4242)
        sel = [it for it in DEV_ITEMS if it["bucket"] in ("unknown_fact", "distractor")]
        p2 = [ET.encode(t2_prompt(it, rng)) for it in sel]
        t2 = [ET.decode(g) for g in greedy_generate(model, p2, CFG["gen_new"], CFG["gen_bs"])]
        acc = defaultdict(lambda: [0, 0])
        for it, t in zip(sel, t2):
            acc[it["bucket"]][0] += int(score_item(it, t))
            acc[it["bucket"]][1] += 1
        out["T2"] = {b: {"k": v[0], "n": v[1], "rate": v[0] / max(1, v[1]), "wilson": wilson(*v)} for b, v in acc.items()}
    return out


# ------------------------------------------------------------------ held-out perplexity and next-token argmax (engine convention: predict positions 1..N-1)
@torch.no_grad()
def eval_ppl(model, kept_logits_fn, want_argmax_rows):
    was_training = getattr(model, "training", False)
    model.eval()
    nll = np.zeros(HELD.shape[0])
    am = []
    bs = CFG["ppl_bs"]
    for s in range(0, HELD.shape[0], bs):
        ids_p = torch.as_tensor(HELD[s:s + bs], dtype=torch.long)
        lg = kept_logits_fn(ids_p).float()[:, :-1]                      # [B, T-1, V_kept]
        tgt = ids_p[:, 1:].to(lg.device)
        lp = torch.log_softmax(lg, -1)
        nll[s:s + bs] = (-lp.gather(-1, tgt[..., None])[..., 0]).sum(-1).cpu().numpy()
        if s < want_argmax_rows:
            am.append(lg.argmax(-1).cpu().numpy())
    if was_training:
        model.train()
    ntok = HELD.shape[0] * (HELD.shape[1] - 1)
    return float(nll.sum() / ntok), nll, (np.concatenate(am, 0)[:want_argmax_rows] if am else None)


def own_kept_logits(model, head_quant=True):
    def f(ids_p):
        dev = embed_weight(model).device
        with autocast_ctx():
            h = backbone(model)(input_ids=KEPT_IDX[ids_p].to(dev), use_cache=False).last_hidden_state
            return head_logits(model, h, KEPT_IDX, head_quant=head_quant)
    return f


# ------------------------------------------------------------------ the reference: stock HF online-BitLinear forward of the unmodified base checkpoint
def stock_pass(dtype, amp):
    """Evaluate the stock Hugging Face model (its own AutoBitLinear online quantizer, config.quantization_config kept) on the held-out windows."""
    model = None
    try:
        cfg = AutoConfig.from_pretrained(BASE_SRC)
        assert getattr(cfg, "quantization_config", None) is not None, "base config carries no quantization_config"
        model = AutoModelForCausalLM.from_pretrained(BASE_SRC, config=cfg, dtype=dtype, low_cpu_mem_usage=True)
        n_auto = sum(1 for m in model.modules() if type(m).__name__ == "AutoBitLinear")
        assert n_auto == 7 * cfg.num_hidden_layers, f"stock model did not convert its projections to online BitLinear ({n_auto} found)"
        model.eval().to(DEVICE0)

        def stock_logits(ids_p):
            with torch.no_grad(), autocast_ctx(amp):
                return model(input_ids=KEPT_IDX[ids_p].to(DEVICE0), use_cache=False).logits[..., KEPT_IDX.to(DEVICE0)]
        return eval_ppl(model, stock_logits, CFG["parity_seqs"])
    except Exception as e:  # noqa
        print("  stock HF pass", dtype, "failed:", type(e).__name__, str(e)[:160])
        return None
    finally:
        if model is not None:
            del model
        gc.collect()
        if CUDA:
            torch.cuda.empty_cache()


def compute_base_reference():
    """Two references. GATE reference = the stock forward with fp32 master weights under the SAME autocast as this notebook's training forward: it isolates implementation
    differences (the one intended difference is the int8 head-input quantization, CIS point 5). INFORMATIONAL reference = the checkpoint's native bf16 stock forward: it adds the
    bf16-vs-autocast rounding noise, which flips int8 activation roundings and moves next-token argmax on its own."""
    path = os.path.join(WORK, "base_ref.npz")
    key = hashlib.sha256((HELD_SHA + BASE_SRC + str(CFG["parity_seqs"]) + "v2").encode()).hexdigest()
    if os.path.exists(path):
        z = np.load(path)
        if str(z["key"]) == key:
            print("base reference loaded from", path)
            return {"kind": str(z["kind"]), "key": key, "mean_nll": float(z["mean_nll"]), "nll_seq": z["nll_seq"], "argmax": z["argmax"],
                    "native": (float(z["native_mean_nll"]), z["native_argmax"]) if bool(z["has_native"]) else None}
    a = stock_pass(torch.float32, True)
    if a is None:
        print("  falling back to the step-0 snapshot of this notebook's own QAT forward (NOT independent)")
        return {"kind": "own-qat-step0-fallback", "key": key, "mean_nll": None, "nll_seq": None, "argmax": None, "native": None}
    b = stock_pass(torch.bfloat16, False)
    ref = {"kind": "hf-stock-online-bitlinear(fp32 master, same autocast)", "key": key, "mean_nll": a[0], "nll_seq": a[1], "argmax": a[2], "native": (b[0], b[2]) if b else None}
    np.savez(path, kind=ref["kind"], key=key, mean_nll=a[0], nll_seq=a[1], argmax=a[2], has_native=b is not None,
             native_mean_nll=(b[0] if b else 0.0), native_argmax=(b[2] if b else np.zeros(1)))
    return ref


def smoke_model_equivalence():
    """Smoke only (tiny model): fp32 stock HF online-BitLinear logits vs this notebook's QAT forward (head quantization off) on the same ids. A random network has a chaotic
    argmax, so the parity percentage printed by the gates is not informative in smoke; this fp32 comparison is."""
    ref = AutoModelForCausalLM.from_pretrained(BASE_SRC, config=AutoConfig.from_pretrained(BASE_SRC), dtype=torch.float32).eval()
    mine = AutoModelForCausalLM.from_pretrained(BASE_SRC, config=strip_quantization_config(AutoConfig.from_pretrained(BASE_SRC)), dtype=torch.float32).eval()
    wrap_qat(mine)
    ids = KEPT_IDX[torch.as_tensor(HELD[:3], dtype=torch.long)]
    with torch.no_grad():
        a = ref(input_ids=ids, use_cache=False).logits[..., KEPT_IDX].float()
        b = head_logits(mine, backbone(mine)(input_ids=ids, use_cache=False).last_hidden_state, KEPT_IDX, head_quant=False).float()
        c = head_logits(mine, backbone(mine)(input_ids=ids, use_cache=False).last_hidden_state, KEPT_IDX, head_quant=True).float()
    out = {"max_abs_logit_diff_fp32_headquant_off": float((a - b).abs().max()), "max_abs_logit": float(a.abs().max()),
           "argmax_agree_headquant_off": float((a.argmax(-1) == b.argmax(-1)).float().mean()), "argmax_agree_headquant_on": float((a.argmax(-1) == c.argmax(-1)).float().mean())}
    # not bit-exact by nature: a 1e-6 summation-order difference can flip one int8 rounding of an activation and the flip propagates (layer 0 matches to ~1e-5, the layer-level test above to 0.0)
    assert out["max_abs_logit_diff_fp32_headquant_off"] <= 0.05 * max(1.0, out["max_abs_logit"]) and out["argmax_agree_headquant_off"] >= 0.95, out
    return out


if SMOKE:
    SMOKE_EQ = smoke_model_equivalence()
    print("[smoke] stock HF online-BitLinear vs this QAT forward, whole tiny model, fp32:", json.dumps(SMOKE_EQ))

BASE = compute_base_reference()
if BASE["mean_nll"] is not None:
    print(f"E1_BASE_REFERENCE kind={BASE['kind']} held-out PPL {math.exp(BASE['mean_nll']):.4f} over {HELD.shape[0]}x{HELD.shape[1] - 1} tokens (identical ids, served-vocabulary softmax)")
    if BASE["native"]:
        print(f"E1_BASE_REFERENCE_NATIVE bf16 stock forward: held-out PPL {math.exp(BASE['native'][0]):.4f} (informational)")

held-out slice: 8 windows x 64 engine-port tokens, source: synthetic prose (fallback; not comparable with the program anchor), sha256 e5468c4d24af71b2...


[smoke] stock HF online-BitLinear vs this QAT forward, whole tiny model, fp32: {"max_abs_logit_diff_fp32_headquant_off": 5.245208740234375e-06, "max_abs_logit": 13.163392066955566, "argmax_agree_headquant_off": 1.0, "argmax_agree_headquant_on": 0.984375}


E1_BASE_REFERENCE kind=hf-stock-online-bitlinear(fp32 master, same autocast) held-out PPL 1042322.4762 over 8x63 tokens (identical ids, served-vocabulary softmax)
E1_BASE_REFERENCE_NATIVE bf16 stock forward: held-out PPL 1043004.7028 (informational)


In [8]:
# ---------------------------------------------------------------- model construction, arms, optimizer, checkpoints
from safetensors.torch import save_file, load_file

ARM_NAMES = {"P": "periphery-only (ternary weights frozen; embeddings, norms, SubLN gains train)", "L": "full QAT, LR 3e-5", "H": "full QAT, LR 1e-4"}
CFG_BASE = AutoConfig.from_pretrained(BASE_SRC)
HID, FF, NL = CFG_BASE.hidden_size, CFG_BASE.intermediate_size, CFG_BASE.num_hidden_layers
KVD = (HID // CFG_BASE.num_attention_heads) * CFG_BASE.num_key_value_heads
N_BODY = NL * (2 * HID * HID + 2 * KVD * HID + 3 * FF * HID)
N_EMB = CFG_BASE.vocab_size * HID
print(f"base: {NL} layers, hidden {HID}, ffn {FF}, ternary params {N_BODY:,}, embedding rows {CFG_BASE.vocab_size}, embedding params {N_EMB:,}")


def is_periphery(name):
    return ("embed_tokens" in name) or ("norm" in name)


def configure_arm(model, arm):
    for n, p in model.named_parameters():
        p.requires_grad = (arm in ("L", "H")) or is_periphery(n)
    return [p for p in model.parameters() if p.requires_grad]


def gib(x):
    return x / 2 ** 30


def plan_arm(arm):
    """Memory arithmetic BEFORE the arm starts, so a card that cannot hold it is reported instead of crashing hours in."""
    opt_b = 2 if (bnb is not None and CUDA) else 8                      # 8-bit Adam state vs fp32 Adam (m and v)
    act = 3 * 2 ** 30
    if arm == "P":
        need = N_BODY * 2 + N_EMB * (4 + 4 + opt_b) + act
        avail = torch.cuda.mem_get_info(0)[0] if CUDA else float("inf")
        where = "one GPU" if CUDA else "cpu"
    else:
        need = (N_BODY + N_EMB) * (4 + 4 + opt_b) + act
        avail = sum(torch.cuda.mem_get_info(i)[0] for i in range(N_GPU)) if CUDA else float("inf")
        where = f"{N_GPU} GPU(s), device_map=auto" if CUDA else "cpu"
    ok = (need <= 0.92 * avail) and (opt_b == 2 or arm == "P" or not CUDA)
    return {"arm": arm, "ok": bool(ok), "need_gib": round(gib(need), 1), "avail_gib": None if avail == float("inf") else round(gib(avail), 1), "where": where,
            "optimizer": "bitsandbytes PagedAdamW8bit" if (bnb is not None and CUDA) else "torch AdamW", "reason": "" if ok else
            f"needs about {gib(need):.1f} GiB across {where} but {gib(avail):.1f} GiB are free (fp32 master weights + fp32 grads + optimizer state); use 2 x T4 or a larger card"}


def build_model_for(arm):
    cfg = strip_quantization_config(AutoConfig.from_pretrained(BASE_SRC))
    if CUDA and arm in ("L", "H"):
        model = AutoModelForCausalLM.from_pretrained(BASE_SRC, config=cfg, dtype=torch.float32, device_map="auto", low_cpu_mem_usage=True)
        wrap_qat(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(BASE_SRC, config=cfg, dtype=torch.bfloat16 if CUDA else torch.float32, low_cpu_mem_usage=True)
        wrap_qat(model)
        if arm == "P":      # frozen ternary weights: fp16 is a lossless container for the bf16 master (8-bit mantissa); everything trainable is fp32
            for m in model.modules():
                if isinstance(m, QATLinear):
                    m.weight.data = m.weight.data.to(torch.float16)
        for n, p in model.named_parameters():
            if is_periphery(n):
                p.data = p.data.float()
        if not CUDA and arm != "P":
            model.float()
        model.to(DEVICE0)
    model.config.use_cache = False
    return model


@torch.no_grad()
def snapshot_ternary(model):
    return {m.qname: snap_ternary(m.weight.float(), weight_ternary_gamma(m.weight.float())).to(torch.int8).cpu() for m in model.modules() if isinstance(m, QATLinear)}


@torch.no_grad()
def ternary_stats(model, init):
    tot = zeros = flips = 0
    per, kinds = {}, defaultdict(lambda: [0, 0])
    for m in model.modules():
        if not isinstance(m, QATLinear):
            continue
        w = m.weight.float()
        q = snap_ternary(w, weight_ternary_gamma(w)).to(torch.int8)
        n = q.numel()
        tot += n
        zeros += int((q == 0).sum())
        if init is not None:
            f = int((q != init[m.qname].to(q.device)).sum())
            flips += f
            per[m.qname] = f / n
            k = m.qname.split(".")[-1]
            kinds[k][0] += f
            kinds[k][1] += n
    return {"p0_census": zeros / tot, "flip_fraction": (flips / tot) if init is not None else 0.0,
            "flip_fraction_max_tensor": max(per.values()) if per else 0.0,
            "flip_fraction_by_projection": {k: v[0] / v[1] for k, v in kinds.items()}}


def make_optimizer(params, lr):
    if CUDA and bnb is not None:
        return bnb.optim.PagedAdamW8bit(params, lr=lr, betas=CFG["betas"], weight_decay=CFG["wd"])
    return torch.optim.AdamW(params, lr=lr, betas=CFG["betas"], weight_decay=CFG["wd"])


def lr_at(step, total, peak):
    w = CFG["warmup"]
    if step < w:
        return peak * (step + 1) / w
    prog = (step - w) / max(1, total - w)
    return peak * (CFG["min_lr_frac"] + (1 - CFG["min_lr_frac"]) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))


# ---- checkpoints: trainable tensors (fp32) + optimizer state + step; two slots so a crash while writing never destroys the last good one
def ckpt_root(arm):
    return os.path.join(WORK, "ckpt", arm)


def save_ckpt(arm, step, model, opt, history):
    root = ckpt_root(arm)
    os.makedirs(root, exist_ok=True)
    slot = os.path.join(root, f"slot_{step}")
    os.makedirs(slot, exist_ok=True)
    sd = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    need = sum(v.numel() * 4 for v in sd.values())
    torch.save(sd, os.path.join(slot, "trainable.pt"))
    free = shutil.disk_usage(WORK).free
    opt_saved = False
    try:
        if free > need * 1.2 + 2 ** 30:
            torch.save(opt.state_dict(), os.path.join(slot, "opt.pt"))
            opt_saved = True
    except Exception as e:  # noqa
        print("optimizer state not saved:", type(e).__name__)
    json.dump({"step": step, "opt_saved": opt_saved, "history": history}, open(os.path.join(slot, "state.json"), "w"))
    json.dump({"latest": f"slot_{step}"}, open(os.path.join(root, "latest.json"), "w"))
    for d in os.listdir(root):
        if d.startswith("slot_") and d != f"slot_{step}":
            shutil.rmtree(os.path.join(root, d), ignore_errors=True)
    del sd


def load_ckpt(arm, model, opt):
    roots = [ckpt_root(arm)]
    if os.environ.get("E1_RESUME_FROM"):
        roots.append(os.path.join(os.environ["E1_RESUME_FROM"], "ckpt", arm))
    for root in roots:
        lp = os.path.join(root, "latest.json")
        if not os.path.exists(lp):
            continue
        slot = os.path.join(root, json.load(open(lp))["latest"])
        st = json.load(open(os.path.join(slot, "state.json")))
        sd = torch.load(os.path.join(slot, "trainable.pt"), map_location="cpu")
        params = dict(model.named_parameters())
        for n, v in sd.items():
            params[n].data.copy_(v.to(params[n].device, params[n].dtype))
        if st.get("opt_saved") and os.path.exists(os.path.join(slot, "opt.pt")):
            opt.load_state_dict(torch.load(os.path.join(slot, "opt.pt"), map_location="cpu"))
        else:
            print("  resume: optimizer state was not saved; Adam moments restart from zero")
        print(f"  resumed arm {arm} from {slot} at step {st['step']}")
        return st["step"], st["history"]
    return 0, []


# ---- export tensors (the E16 'unpacked ternary' layout the repacker accepts). MULTIPLY convention: weight_scale = gamma = mean|w| (F32, shape [1]).
@torch.no_grad()
def export_tensors(model):
    t = {}
    for m in model.modules():
        if isinstance(m, QATLinear):
            w = m.weight.float()
            g = weight_ternary_gamma(w)
            t[f"{m.qname}.weight"] = snap_ternary(w, g).to(torch.int8).cpu().contiguous()
            t[f"{m.qname}.weight_scale"] = g.reshape(1).float().cpu().contiguous()
    for n, p in model.named_parameters():
        if is_periphery(n):
            t[n] = p.detach().to(torch.bfloat16).cpu().contiguous()           # the engine reads embeddings and norms as BF16 (round-to-nearest-even here)
    assert len([k for k in t if k.endswith(".weight_scale")]) == 7 * NL
    return t


# ---------------------------------------------------------------- data order and the loss
PERMS = {}


def rows_for_step(step):
    per = CFG["mb"] * CFG["accum"]
    N = TRAIN_X.shape[0]
    out = []
    for j in range(per):
        c = step * per + j
        ep, k = divmod(c, N)
        if ep not in PERMS:
            PERMS[ep] = np.random.RandomState(SEED + ep).permutation(N)
        out.append(int(PERMS[ep][k]))
    return out


def lm_loss(model, x_np, m_np):
    """assistant-only cross-entropy over the served ids; the head input is int8-quantized (CIS point 5). Returns (sum of NLL, number of loss tokens)."""
    dev = embed_weight(model).device
    inp = KEPT_IDX[torch.as_tensor(x_np[:, :-1], dtype=torch.long)].to(dev)
    tgt = torch.as_tensor(x_np[:, 1:], dtype=torch.long)
    sel = torch.as_tensor(m_np[:, 1:], dtype=torch.bool)
    h = backbone(model)(input_ids=inp, use_cache=False).last_hidden_state
    h_sel = h[sel.to(h.device)]
    logits = head_logits(model, h_sel, KEPT_IDX)
    return F.cross_entropy(logits.float(), tgt[sel].to(logits.device), reduction="sum"), int(sel.sum())


def gate_block(arm, step, total, rec, first_rec):
    p0 = rec["p0_unknown_strict"]
    dis = rec["parity_disagreement"]
    tag = "GATE at step 0" if step == 0 else "drift, informational after step 0"
    ppl_flag = "FLAG >1.02" if rec["ppl_ratio"] > 1.02 else "ok"
    d0 = first_rec["P0"].get("distractor", {"rate": float("nan")})["rate"] if first_rec else rec["P0"].get("distractor", {"rate": float("nan")})["rate"]
    dd = rec["P0"].get("distractor", {"k": 0, "n": 0, "rate": float("nan"), "wilson": (0, 0)})
    lines = [f"=== arm {arm} | step {step}/{total} | {'SMOKE plumbing check' if SMOKE else 'measurement'} ===",
             f"  parity vs base (argmax disagreement) : {dis:.2%}   [<= 3.00% {tag}: {'PASS' if dis <= 0.03 else 'FAIL'}]",
             f"  held-out PPL ratio vs base           : {rec['ppl_ratio']:.4f}   [<= 1.05 GATE: {'PASS' if rec['ppl_ratio'] <= 1.05 else 'FAIL (arm stops)'}; {ppl_flag}]",
             f"  P0 unknown_fact, begins LOOKUP(<key>): {p0['k']}/{p0['n']} = {p0['rate']:.1%} (Wilson {p0['wilson'][0]:.0%}-{p0['wilson'][1]:.0%})   [>= 73.3% GATE: {'PASS' if p0['rate'] >= 0.733 else 'FAIL'}]",
             f"  distractor no-tool precision         : {dd['k']}/{dd['n']} = {dd['rate']:.1%}   [>= step-0 value {d0:.1%}: {'PASS' if dd['rate'] + 1e-9 >= d0 else 'FAIL'}]",
             "  dev buckets (call correct / no call): " + " | ".join(f"{b} {v['k']}/{v['n']}" for b, v in sorted(rec["P0"].items()))]
    if "parity_disagreement_headquant_off" in rec:
        lines.append(f"  (attribution) same comparison with the head-input quantization switched off: {rec['parity_disagreement_headquant_off']:.2%} (the stock forward has no CIS point 5)")
    if "native_bf16_reference" in rec:
        n_ = rec["native_bf16_reference"]
        lines.append(f"  (informational) vs the checkpoint's native bf16 stock forward: PPL ratio {n_['ppl_ratio']:.4f}, argmax disagreement {n_['parity_disagreement']:.2%} - includes bf16-vs-autocast rounding noise")
    if "T2" in rec:
        lines.append("  T2 (2 LOOKUP shots): " + " | ".join(f"{b} {v['k']}/{v['n']}" for b, v in sorted(rec["T2"].items())))
    if rec.get("ternary"):
        t = rec["ternary"]
        lines.append(f"  p(0) census {t['p0_census']:.4f} (incumbent 0.42) | trit-flip fraction {t['flip_fraction']:.2e} (max tensor {t['flip_fraction_max_tensor']:.2e})")
    print("\n".join(lines))


def full_eval(model, arm, step, total, init_tern, first_rec, with_t2=False):
    mean_nll, nll, am = eval_ppl(model, own_kept_logits(model), CFG["parity_seqs"])
    rec = {"step": step, "heldout_ppl": math.exp(mean_nll), "ppl_ratio": math.exp(mean_nll - BASE["mean_nll"]),
           "parity_disagreement": float((am != BASE["argmax"]).mean())}
    if step == 0:        # attribution: how much of the disagreement is the int8 head-input quantization (CIS point 5) that the stock forward does not have
        _, _, am_off = eval_ppl(model, own_kept_logits(model, head_quant=False), CFG["parity_seqs"])
        rec["parity_disagreement_headquant_off"] = float((am_off != BASE["argmax"]).mean())
    if BASE.get("native") and step == 0:
        rec["native_bf16_reference"] = {"ppl_ratio": math.exp(mean_nll - BASE["native"][0]), "parity_disagreement": float((am != BASE["native"][1]).mean())}
    rec.update(eval_dev(model, with_t2=with_t2))
    rec["ternary"] = ternary_stats(model, init_tern) if arm != "P" or step == 0 else {"p0_census": first_rec["ternary"]["p0_census"] if first_rec else None, "flip_fraction": 0.0,
                                                                                     "flip_fraction_max_tensor": 0.0, "flip_fraction_by_projection": {}}
    gate_block(arm, step, total, rec, first_rec)
    return rec


# ---------------------------------------------------------------- one arm
ARM_DIR = os.path.join(WORK, "arms")
os.makedirs(ARM_DIR, exist_ok=True)
os.makedirs(os.path.join(WORK, "logs"), exist_ok=True)


def arm_result_path(arm):
    return os.path.join(ARM_DIR, arm, "result.json")


def train_arm(arm, stop_after=None):
    rp = arm_result_path(arm)
    os.makedirs(os.path.dirname(rp), exist_ok=True)
    if os.path.exists(rp):
        prev = json.load(open(rp))
        if prev["status"] in ("COMPLETE", "KILLED", "SKIPPED"):
            print(f"arm {arm}: already {prev['status']} (result.json present), not re-running")
            return prev
    plan = plan_arm(arm)
    print(f"\n########## ARM {arm}: {ARM_NAMES[arm]} | lr {CFG['lr'][arm]} | plan {json.dumps(plan)}")
    if not plan["ok"]:
        res = {"arm": arm, "status": "SKIPPED", "reason": plan["reason"], "plan": plan}
        json.dump(res, open(rp, "w"), indent=1)
        print("  SKIPPED:", plan["reason"])
        return res
    per_step = CFG["mb"] * CFG["accum"]
    total = TRAIN_X.shape[0] // per_step
    if CFG["max_steps"]:
        total = min(total, CFG["max_steps"])
    peak = CFG["lr"][arm]
    model = build_model_for(arm)
    init_tern = snapshot_ternary(model) if arm != "P" else None
    params = configure_arm(model, arm)
    n_train = sum(p.numel() for p in params)
    print(f"  trainable parameters {n_train:,} of {sum(p.numel() for p in model.parameters()):,}; steps {total} x {per_step * CFG['seq']:,} packed tokens")
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.train()
    opt = make_optimizer(params, peak)
    scaler = torch.amp.GradScaler("cuda") if USE_SCALER else None
    start, history = load_ckpt(arm, model, opt)
    if BASE["mean_nll"] is None:       # fallback reference (see compute_base_reference): this model's own step-0 forward without the head quantization
        BASE["mean_nll"], _, BASE["argmax"] = eval_ppl(model, own_kept_logits(model, head_quant=False), CFG["parity_seqs"])
        print("  fallback reference taken from the step-0 snapshot of this notebook's forward (not independent)")
    if start == 0:
        assert generation_selfcheck_ok(model), "batched generation disagrees with single-sample generation"
    evals = [h for h in history if "ppl_ratio" in h]
    first_rec = evals[0] if evals else None
    logf = open(os.path.join(WORK, "logs", f"{arm}.jsonl"), "a")
    status, killed = "RUNNING", None
    if start == 0:
        s0 = os.path.join(WORK, "export", "step0")
        if not os.path.exists(os.path.join(s0, "model.safetensors")):          # the unmodified incumbent, exported by the same code: the K12 check target
            os.makedirs(s0, exist_ok=True)
            save_file(export_tensors(model), os.path.join(s0, "model.safetensors"), metadata={"format": "pt"})
            _, nll0, am0 = eval_ppl(model, own_kept_logits(model), CFG["parity_seqs"])
            np.save(os.path.join(s0, "final_argmax.npy"), am0)
            np.save(os.path.join(s0, "final_nll.npy"), nll0)
            print(f"  step-0 export of the incumbent written to {s0}")
        first_rec = full_eval(model, arm, 0, total, init_tern, None, with_t2=True)
        first_rec["note"] = "step 0 = the incumbent through this notebook's QAT forward (baseline in the same harness)"
        evals.append(first_rec)
        history.append(first_rec)
        logf.write(json.dumps(first_rec) + "\n")
        if first_rec["parity_disagreement"] > 0.03:
            print("  WARNING: step-0 parity gate FAILED (> 3% argmax disagreement vs the base reference)." + (
                " [smoke: a random network has a chaotic argmax; the exact fp32 comparison above is the informative one]" if SMOKE else
                " Stop and investigate before trusting any number from this arm (scale convention? activation points? reference dtype?)."))
    t_tok, t_time, losses = 0, 0.0, [h["loss"] for h in history if "loss" in h and "ppl_ratio" not in h]
    tps_log = [h["tokens_per_s"] for h in history if "tokens_per_s" in h]
    for step in range(start, total):
        if stop_after is not None and step >= stop_after:
            save_ckpt(arm, step, model, opt, history)
            status = "PAUSED:test-stop"
            break
        if time.time() - T_SESSION > CFG["max_hours"] * 3600:
            save_ckpt(arm, step, model, opt, history)
            status = "PAUSED:session-time"
            print(f"  session time budget reached at step {step}; checkpoint written, re-run the notebook to resume")
            break
        lr = lr_at(step, total, peak)
        for g in opt.param_groups:
            g["lr"] = lr
        rows = rows_for_step(step)
        n_tot = int(sum(TRAIN_M[r, 1:].sum() for r in rows))
        if CUDA:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        opt.zero_grad(set_to_none=True)
        loss_sum = 0.0
        for a in range(CFG["accum"]):
            r = rows[a * CFG["mb"]:(a + 1) * CFG["mb"]]
            with autocast_ctx():
                ls, n = lm_loss(model, TRAIN_X[r], TRAIN_M[r])
                loss = ls / n_tot
            (scaler.scale(loss) if scaler else loss).backward()
            loss_sum += float(ls.detach())
        if scaler:
            scaler.unscale_(opt)
        gn = float(torch.nn.utils.clip_grad_norm_(params, CFG["clip"]))
        finite = math.isfinite(loss_sum) and math.isfinite(gn)
        if finite:
            (scaler.step(opt) if scaler else opt.step())
            if scaler:
                scaler.update()
        if CUDA:
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        loss_mean = loss_sum / max(1, n_tot)
        if not finite:
            if scaler and not math.isfinite(gn) and math.isfinite(loss_sum):
                scaler.update()                      # fp16 overflow in the grads: GradScaler skips the step and lowers its scale (not a NaN loss)
                continue
            status, killed = "KILLED", "non-finite loss"
            print(f"  KILL GATE: non-finite loss at step {step}")
            break
        t_tok += per_step * CFG["seq"]
        t_time += dt
        losses.append(loss_mean)
        log_every = 5 if SMOKE else 10
        if (step + 1) % log_every == 0 or step == start:
            tps = per_step * CFG["seq"] / dt
            tps_log.append(tps)
            rec = {"step": step + 1, "loss": loss_mean, "lr": lr, "grad_norm": gn, "tokens_per_s": tps, "tokens_seen": (step + 1) * per_step * CFG["seq"]}
            if CUDA:
                rec["peak_mem_gib"] = [round(torch.cuda.max_memory_allocated(i) / 2 ** 30, 2) for i in range(N_GPU)]
            history.append(rec)
            logf.write(json.dumps(rec) + "\n")
            logf.flush()
            print(f"  step {step + 1:5d}/{total} loss {loss_mean:.4f} lr {lr:.2e} gnorm {gn:.2f} packed_tokens_per_s {tps:.1f}")
        if (step + 1) % CFG["eval_every"] == 0 or step + 1 == total:
            rec = full_eval(model, arm, step + 1, total, init_tern, first_rec, with_t2=(step + 1 == total))
            evals.append(rec)
            history.append(rec)
            logf.write(json.dumps(rec) + "\n")
            logf.flush()
            if rec["ppl_ratio"] > 1.05:
                status, killed = "KILLED", f"held-out PPL ratio {rec['ppl_ratio']:.3f} > 1.05"
                print("  KILL GATE:", killed)
                break
        if (step + 1) % CFG["ckpt_every"] == 0 and step + 1 < total:
            save_ckpt(arm, step + 1, model, opt, history)
    else:
        status = "COMPLETE"
    steps_done = (history[-1]["step"] if history else 0)
    lw = max(1, min(10, len(losses) // 3))                       # compare the first and last windows (10 steps; fewer when the run is short, e.g. smoke)
    res = {"arm": arm, "name": ARM_NAMES[arm], "lr": peak, "status": status, "killed_by": killed, "total_steps": total, "steps_done": steps_done,
           "trainable_params": n_train, "plan": plan, "optimizer": plan["optimizer"], "baseline_step0": first_rec, "evals": evals,
           "final": evals[-1] if evals else None, "loss_window": lw, "loss_first10": statistics.mean(losses[:lw]) if losses else None,
           "loss_last10": statistics.mean(losses[-lw:]) if losses else None, "loss_decreased": bool(losses and statistics.mean(losses[-lw:]) < statistics.mean(losses[:lw])),
           "train_tokens_per_s_median": statistics.median(tps_log) if tps_log else None, "data_sha256": man["files"]["train_seqs.npy"] if "train_seqs.npy" in man["files"] else None,
           "heldout_source": HELD_SRC, "reference_kind": BASE["kind"]}
    if status in ("COMPLETE", "KILLED") and evals:
        exp_dir = os.path.join(WORK, "export", arm)
        os.makedirs(exp_dir, exist_ok=True)
        save_file(export_tensors(model), os.path.join(exp_dir, "model.safetensors"), metadata={"format": "pt"})
        _, nll_final, am_final = eval_ppl(model, own_kept_logits(model), CFG["parity_seqs"])
        np.save(os.path.join(exp_dir, "final_argmax.npy"), am_final)
        np.save(os.path.join(exp_dir, "final_nll.npy"), nll_final)
        res["export_ppl"] = evals[-1]["heldout_ppl"]
        if os.environ.get("E1_KEEP_CKPT") != "1":
            shutil.rmtree(ckpt_root(arm), ignore_errors=True)
    if status.startswith("PAUSED"):
        res_tmp = dict(res)
        json.dump(res_tmp, open(rp + ".partial", "w"), indent=1, default=str)
    else:
        json.dump(res, open(rp, "w"), indent=1, default=str)
    logf.close()
    print(f"ARM {arm} -> {status}" + (f" ({killed})" if killed else ""))
    del opt, model
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()
    return res


def generation_selfcheck_ok(model):
    same = generation_selfcheck(model)
    thr = 0.99        # fp32 comparison: any mask / position-id / cache bug in left-padded batching shows up as a large drop
    print(f"  generation self-check (batched left-padded vs single-sample, 8 tokens x 6 prompts): {same:.1%} identical tokens (>= {thr:.0%} required)")
    return same >= thr

base: 2 layers, hidden 64, ffn 128, ternary params 73,728, embedding rows 128256, embedding params 8,208,384


In [9]:
ARM_RESULTS = {}
for arm in ARMS:
    if SMOKE and arm == ARMS[0]:
        # prove checkpoint/resume: stop at the first checkpoint, then resume the same arm from disk
        part = train_arm(arm, stop_after=CFG["ckpt_every"])
        assert part["status"] == "PAUSED:test-stop", part["status"]
        print(f"[smoke] resume test: arm {arm} paused at its checkpoint; resuming from disk")
    ARM_RESULTS[arm] = train_arm(arm)
print("\narm status:", {a: r["status"] for a, r in ARM_RESULTS.items()})


########## ARM P: periphery-only (ternary weights frozen; embeddings, norms, SubLN gains train) | lr 0.003 | plan {"arm": "P", "ok": true, "need_gib": 3.1, "avail_gib": null, "where": "cpu", "optimizer": "torch AdamW", "reason": ""}
  trainable parameters 8,209,088 of 8,282,816; steps 12 x 512 packed tokens


  generation self-check (batched left-padded vs single-sample, 8 tokens x 6 prompts): 100.0% identical tokens (>= 99% required)


  step-0 export of the incumbent written to e1_smoke_out/export/step0


=== arm P | step 0/12 | SMOKE plumbing check ===
  parity vs base (argmax disagreement) : 2.38%   [<= 3.00% GATE at step 0: PASS]
  held-out PPL ratio vs base           : 0.9996   [<= 1.05 GATE: PASS; ok]
  P0 unknown_fact, begins LOOKUP(<key>): 0/4 = 0.0% (Wilson 0%-49%)   [>= 73.3% GATE: FAIL]
  distractor no-tool precision         : 4/4 = 100.0%   [>= step-0 value 100.0%: PASS]
  dev buckets (call correct / no call): calc_easy 0/4 | calc_hard 0/4 | calc_overflow 0/4 | distractor 4/4 | file_read 0/4 | in_context_answerable 4/4 | lookup_hit 0/4 | lookup_miss 0/4 | lookup_near_miss 0/4 | unanswerable 2/2 | undeclared_tool 4/4 | unknown_fact 0/4
  (attribution) same comparison with the head-input quantization switched off: 0.00% (the stock forward has no CIS point 5)
  (informational) vs the checkpoint's native bf16 stock forward: PPL ratio 0.9989, argmax disagreement 22.62% - includes bf16-vs-autocast rounding noise
  T2 (2 LOOKUP shots): distractor 4/4 | unknown_fact 0/4
  p(0) census

  step     1/12 loss 13.7218 lr 1.50e-03 gnorm 18.66 packed_tokens_per_s 1504.7


...

  step     5/12 loss 13.4257 lr 2.74e-03 gnorm 20.19 packed_tokens_per_s 1959.9


=== arm H | step 6/12 | SMOKE plumbing check ===
  parity vs base (argmax disagreement) : 87.30%   [<= 3.00% drift, informational after step 0: FAIL]
  held-out PPL ratio vs base           : 0.7959   [<= 1.05 GATE: PASS; ok]
  P0 unknown_fact, begins LOOKUP(<key>): 0/4 = 0.0% (Wilson 0%-49%)   [>= 73.3% GATE: FAIL]
  distractor no-tool precision         : 4/4 = 100.0%   [>= step-0 value 100.0%: PASS]
  dev buckets (call correct / no call): calc_easy 0/4 | calc_hard 0/4 | calc_overflow 0/4 | distractor 4/4 | file_read 0/4 | in_context_answerable 4/4 | lookup_hit 0/4 | lookup_miss 0/4 | lookup_near_miss 0/4 | unanswerable 2/2 | undeclared_tool 4/4 | unknown_fact 0/4
  p(0) census 0.3085 (incumbent 0.42) | trit-flip fraction 5.97e-03 (max tensor 7.45e-03)


  step    10/12 loss 13.4469 lr 8.56e-04 gnorm 12.99 packed_tokens_per_s 2153.9


=== arm H | step 12/12 | SMOKE plumbing check ===
  parity vs base (argmax disagreement) : 88.49%   [<= 3.00% drift, informational after step 0: FAIL]
  held-out PPL ratio vs base           : 0.7031   [<= 1.05 GATE: PASS; ok]
  P0 unknown_fact, begins LOOKUP(<key>): 0/4 = 0.0% (Wilson 0%-49%)   [>= 73.3% GATE: FAIL]
  distractor no-tool precision         : 4/4 = 100.0%   [>= step-0 value 100.0%: PASS]
  dev buckets (call correct / no call): calc_easy 0/4 | calc_hard 0/4 | calc_overflow 0/4 | distractor 4/4 | file_read 0/4 | in_context_answerable 4/4 | lookup_hit 0/4 | lookup_miss 0/4 | lookup_near_miss 0/4 | unanswerable 2/2 | undeclared_tool 4/4 | unknown_fact 0/4
  T2 (2 LOOKUP shots): distractor 4/4 | unknown_fact 0/4
  p(0) census 0.3086 (incumbent 0.42) | trit-flip fraction 7.15e-03 (max tensor 8.67e-03)


ARM H -> COMPLETE

arm status: {'P': 'COMPLETE', 'L': 'COMPLETE', 'H': 'COMPLETE'}


## Cell 4 - export in the layout `aegis-forge/repack_ternary.py` accepts

**Layout** (`--source-packing unpacked`, the E16 form): `model.safetensors` with, for each of the 7 projections of every layer, `…proj.weight` as **int8 `{-1,0,+1}`** and `…proj.weight_scale` as **F32 `[1]`**;
RMSNorm / SubLN gains and `model.embed_tokens.weight` (the full 128,256-row table; the repacker slices the 50,256 served rows itself) as **BF16**; `config.json`, `tokenizer.json`, `tokenizer_config.json`.

**Scale convention chosen: MULTIPLY, `weight_scale = gamma = mean|w|` (the engine multiplies the ternary dot by the stored scalar), and the command pins `--scale-convention multiply`.** Why:
1. It is the incumbent's own convention. Measured while preparing this notebook (range-fetch of three tensors, bf16 master vs the packed `microsoft/bitnet-b1.58-2B-4T`): L0 `q_proj` mean|w| = 1.218851 vs stored `weight_scale` 1.21875; L0 `down_proj` 2.163143 vs 2.15625; L15 `gate_proj` 2.173966 vs 2.171875
   (stored values are `gamma` rounded to bf16). The reciprocal would store about 0.46 and every linear would be about 4.7x too small - the K12 failure mode (LAB-02: "decodes to `, the and the and ...` and passes every shape/ternary check").
2. `repack_ternary.py --scale-convention auto` resolves to `multiply` iff `config.json` carries `quantization_config.quant_method == "bitnet"`. The base config has it and this export keeps it, so even a forgotten flag does the right thing;
   the tinybit divide convention (`weight_scale = 1/gamma`) would instead only be safe with that key *removed*, and any later re-save of the config through transformers would silently flip it.
3. The flag is pinned in the printed command anyway, and `validate_export` re-derives the repacker's `auto` decision from the written `config.json` and asserts it equals the convention used.
The scale is stored as full F32 `gamma` (the incumbent stores it rounded to bf16), so a step-0 export is *not* expected to reproduce the incumbent's digest `cab11400d737ac4a`.

`export/step0` is the **unmodified incumbent pushed through this same export code** (written before the first update of the first arm): repack and measure it FIRST on the Linux side, it is the K12 scale-convention check (engine PPL vs the HF-side PPL printed here).
Step-0 export parity (gate 5) is run here by reloading the written file into a fresh model (ternary x scale, BF16-rounded embeddings and norms, as the engine reads them) and comparing it with the trained model it came from.
Engine parity (HF-vs-engine PPL gap <= 3%, `cis_decode` digest repeatability, receipts) can only be produced on the Linux side; the commands are printed below.

In [10]:
EXPORT_ROOT = os.path.join(WORK, "export")
PINNED_CONVENTION = "multiply"
REPACKER_CANDIDATES = [os.environ.get("E1_REPACKER", ""), "/home/user/aefinity-ai/alice-aegis/aegis-forge/repack_ternary.py"]


def export_config_dict():
    rt = getattr(CFG_BASE, "rope_theta", None)
    if rt is None:
        rt = (getattr(CFG_BASE, "rope_parameters", None) or {}).get("rope_theta", 500000.0)
    return {"architectures": ["BitNetForCausalLM"], "model_type": "bitnet", "hidden_size": HID, "intermediate_size": FF, "num_hidden_layers": NL,
            "num_attention_heads": CFG_BASE.num_attention_heads, "num_key_value_heads": CFG_BASE.num_key_value_heads, "hidden_act": "relu2",
            "max_position_embeddings": CFG_BASE.max_position_embeddings, "rms_norm_eps": float(CFG_BASE.rms_norm_eps), "rope_theta": float(rt),
            "tie_word_embeddings": True, "vocab_size": FULL_VOCAB, "bos_token_id": CFG_BASE.bos_token_id, "eos_token_id": CFG_BASE.eos_token_id,
            "torch_dtype": "float32", "quantization_config": {"quant_method": "bitnet", "linear_class": "autobitlinear", "quantization_mode": "offline"}}


def resolve_scale_convention_like_repacker(cfg, arg="auto"):
    """Verbatim logic of repack_ternary.resolve_scale_convention."""
    if arg != "auto":
        return arg
    return "multiply" if (cfg.get("quantization_config") or {}).get("quant_method") == "bitnet" else "reciprocal"


def complete_export_dir(arm):
    d = os.path.join(EXPORT_ROOT, arm)
    assert os.path.exists(os.path.join(d, "model.safetensors")), f"arm {arm} has no model.safetensors (did it finish?)"
    json.dump(export_config_dict(), open(os.path.join(d, "config.json"), "w"), indent=2)
    shutil.copy(TOK_JSON, os.path.join(d, "tokenizer.json"))
    if TOK_CFG:
        shutil.copy(TOK_CFG, os.path.join(d, "tokenizer_config.json"))
    contract = {"layout": "unpacked ternary: I8 {-1,0,+1} weights + F32 [1] weight_scale per projection; BF16 norms and full embedding table",
                "weight_scale_convention": PINNED_CONVENTION, "weight_scale_meaning": "gamma = mean|w_latent| (engine multiplies the ternary dot by it)",
                "repack_flag": f"--scale-convention {PINNED_CONVENTION}", "auto_resolves_to": resolve_scale_convention_like_repacker(export_config_dict()),
                "tensors_expected": 7 * NL * 2 + 4 * NL + 2, "arm": arm, "vocab_rows": FULL_VOCAB, "pruned_vocab": ET.vocab_size,
                "engine_port_vocab_bin_sha256": vb_sha, "smoke": SMOKE}
    json.dump(contract, open(os.path.join(d, "EXPORT_CONTRACT.json"), "w"), indent=1)
    return d


def validate_export(d):
    """Independent re-read of the written file against what repack_ternary.py will demand."""
    from safetensors import safe_open
    cfg = json.load(open(os.path.join(d, "config.json")))
    assert resolve_scale_convention_like_repacker(cfg) == PINNED_CONVENTION, "config.json would make `--scale-convention auto` pick the other convention"
    n_w = n_s = n_n = 0
    with safe_open(os.path.join(d, "model.safetensors"), framework="pt") as f:
        keys = set(f.keys())
        for i in range(NL):
            for proj, (o, ii) in {"self_attn.q_proj": (HID, HID), "self_attn.k_proj": (KVD, HID), "self_attn.v_proj": (KVD, HID), "self_attn.o_proj": (HID, HID),
                                  "mlp.gate_proj": (FF, HID), "mlp.up_proj": (FF, HID), "mlp.down_proj": (HID, FF)}.items():
                w = f.get_tensor(f"model.layers.{i}.{proj}.weight")
                s = f.get_tensor(f"model.layers.{i}.{proj}.weight_scale")
                assert w.dtype == torch.int8 and tuple(w.shape) == (o, ii), (proj, w.dtype, w.shape)
                assert int(w.min()) >= -1 and int(w.max()) <= 1, "non-ternary value"
                assert s.dtype == torch.float32 and s.shape == (1,) and float(s) > 0 and math.isfinite(float(s))
                n_w += 1
                n_s += 1
            for nm in ("input_layernorm", "post_attention_layernorm", "self_attn.attn_sub_norm", "mlp.ffn_sub_norm"):
                t = f.get_tensor(f"model.layers.{i}.{nm}.weight")
                assert t.dtype == torch.bfloat16
                n_n += 1
        e = f.get_tensor("model.embed_tokens.weight")
        assert e.dtype == torch.bfloat16 and tuple(e.shape) == (FULL_VOCAB, HID)
        assert f.get_tensor("model.norm.weight").dtype == torch.bfloat16
        assert "lm_head.weight" not in keys, "tie_word_embeddings=True: no lm_head.weight"
        assert len(keys) == 7 * NL * 2 + 4 * NL + 2, len(keys)
    return {"projection_weights": n_w, "scales": n_s, "norms": n_n + 1, "tensors": len(keys)}


@torch.no_grad()
def load_export_model(d):
    """A fresh model whose weights are READ BACK from the exported file: body = int8 x scale (fp16 container), embeddings/norms = BF16-rounded values."""
    from safetensors import safe_open
    model = AutoModelForCausalLM.from_pretrained(BASE_SRC, config=strip_quantization_config(AutoConfig.from_pretrained(BASE_SRC)),
                                                 dtype=torch.float32 if not CUDA else torch.bfloat16, low_cpu_mem_usage=True)
    wrap_qat(model)
    params = dict(model.named_parameters())
    with safe_open(os.path.join(d, "model.safetensors"), framework="pt") as f:
        for m in model.modules():
            if isinstance(m, QATLinear):
                w = f.get_tensor(f"{m.qname}.weight").float()
                s = f.get_tensor(f"{m.qname}.weight_scale").float()
                m.weight.data = (w * s).to(torch.float16)           # MULTIPLY convention: effective weight = w_q * stored scale
                m.exported = True
        for n, p in params.items():
            if is_periphery(n):
                p.data = f.get_tensor(n).float()                    # BF16 values, as the engine reads them
    model.config.use_cache = False
    return model.eval().to(DEVICE0)


def export_parity(arm):
    d = os.path.join(EXPORT_ROOT, arm)
    nll_tr = np.load(os.path.join(d, "final_nll.npy"))
    am_tr = np.load(os.path.join(d, "final_argmax.npy"))
    model = load_export_model(d)
    mean_nll, nll, am = eval_ppl(model, own_kept_logits(model), CFG["parity_seqs"])
    ppl_tr, ppl_ex = math.exp(nll_tr.sum() / (HELD.shape[0] * (HELD.shape[1] - 1))), math.exp(mean_nll)
    # token parity: the tokenizer.json that ships in the export, run through the same port, must give the same VOCAB.BIN bytes and the same ids
    et2 = EngineTokenizer(json.load(open(os.path.join(d, "tokenizer.json"))), cut=TOK_CUT, special_base=TOK_SPECIAL_BASE)
    probe = [episode_text(e) for e in (train_eps[:300] if train_eps else [])] + SKEW_CASES
    tok_ok = (et2.vocab_bin_bytes() == vb) and all(et2.encode(t) == ET.encode(t) for t in probe)
    out = {"ppl_trained": ppl_tr, "ppl_reloaded_export": ppl_ex, "ppl_gap": abs(ppl_ex / ppl_tr - 1), "argmax_agreement_informational": float((am == am_tr).mean()),
           "token_parity_exported_tokenizer": bool(tok_ok), "slice": f"{HELD.shape[0]} windows (PPL) / first {CFG['parity_seqs']} (argmax)"}
    out["pass"] = bool(out["ppl_gap"] <= 0.03 and tok_ok)
    del model
    gc.collect()
    if CUDA:
        torch.cuda.empty_cache()
    return out


def run_repacker(arm, d):
    rp = next((p for p in REPACKER_CANDIDATES if p and os.path.exists(p)), None)
    if rp is None:
        return {"ran": False, "reason": "repack_ternary.py is on the Linux side; not present in this environment"}
    art = os.path.join(WORK, "artifacts", arm)
    cmd = [sys.executable, rp, d, art, "--max-seq", "2048", "--source-packing", "unpacked", "--scale-convention", PINNED_CONVENTION]
    if not FALLBACK_TOKENIZER:
        cmd.insert(4, "--llama3-prune")
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = {"ran": True, "cmd": " ".join(cmd), "rc": r.returncode, "tail": r.stdout.strip().splitlines()[-3:], "stderr": r.stderr.strip()[-300:]}
    if r.returncode == 0:
        from safetensors import safe_open
        with safe_open(os.path.join(d, "model.safetensors"), framework="pt") as fe, safe_open(os.path.join(art, "MODEL.SAF"), framework="pt") as fm:
            k = "model.layers.0.self_attn.q_proj.weight_scale"
            out["scale_exported"], out["scale_in_MODEL_SAF"] = float(fe.get_tensor(k)), float(fm.get_tensor(k))
            out["MODEL_SAF_tensors"] = len(list(fm.keys()))
        out["scale_roundtrip_ok"] = out["scale_exported"] == out["scale_in_MODEL_SAF"]
        out["vocab_bin_sha256"] = sha256_file(os.path.join(art, "VOCAB.BIN"))
        out["vocab_bin_equals_port"] = out["vocab_bin_sha256"] == vb_sha
        out["artifacts"] = art
    return out


def tsv_escape(v):
    assert "\t" not in v
    return v.replace("\\", "\\\\").replace("\n", "\\n")


def write_gateway_suites(d):
    """Dev slice as eval-suite TSVs (the 8-column format of run_suite.sh) plus the merged table, so the SAME items run through the real agent_trace gateway."""
    hdr = ["item_id", "bucket", "prompt_template_id", "prompt_text", "expected_tool", "expected_input", "expected_output", "notes"]
    ok_buckets = {"calc_easy", "calc_hard", "calc_overflow", "lookup_hit", "lookup_miss", "lookup_near_miss", "unknown_fact", "distractor", "in_context_answerable"}
    rng = random.Random(4242)
    paths = {}
    for variant in ("P0", "T2"):
        rows = []
        for i, it in enumerate(DEV_ITEMS):
            if it["bucket"] not in ok_buckets or (variant == "T2" and it["bucket"] not in ("unknown_fact", "distractor")):
                continue
            prompt = it["prompt"] if variant == "P0" else t2_prompt(it, rng)
            tool = it["calls"][0].split("(")[0] if it["calls"] else "NONE"
            rows.append([f"e1dev_{variant}_{i:04d}", it["bucket"] + ("_t2" if variant == "T2" else ""), variant, prompt, tool,
                         it["calls"][0] if it["calls"] else "", it["outputs"][0] if it["outputs"] else "", f"generator={it['generator']} ns={it['ns']}"])
        p = os.path.join(d, f"suite_e1_dev_{variant}.tsv")
        with open(p, "w") as f:
            f.write("\t".join(hdr) + "\n")
            for r in rows:
                f.write("\t".join(tsv_escape(c) for c in r) + "\n")
        paths[variant] = (p, len(rows))
    shutil.copy(os.path.join(DATA_DIR, "dev_table.tsv"), os.path.join(d, "dev_table.tsv"))
    return paths


def sha256_dir(d):
    sums = {}
    for fn in sorted(os.listdir(d)):
        p = os.path.join(d, fn)
        if os.path.isfile(p) and fn != "SHA256SUMS":
            sums[fn] = sha256_file(p)
    with open(os.path.join(d, "SHA256SUMS"), "w") as f:
        for fn, h in sums.items():
            f.write(f"{h}  {fn}\n")
    return sums


EXPORTS = {}
for arm, res in [("step0", {"status": "COMPLETE"})] + list(ARM_RESULTS.items()):
    if res.get("status") not in ("COMPLETE", "KILLED") or not os.path.exists(os.path.join(EXPORT_ROOT, arm, "model.safetensors")):
        print(f"arm {arm}: nothing to export ({res.get('status')})")
        continue
    d = complete_export_dir(arm)
    info = {"dir": d, "validate": validate_export(d)}
    info["parity"] = export_parity(arm)
    info["suites"] = {k: (os.path.basename(p), n) for k, (p, n) in write_gateway_suites(d).items()}
    info["repack_check"] = run_repacker(arm, d)
    info["sha256"] = sha256_dir(d)
    EXPORTS[arm] = info
    print(f"\n--- export arm {arm}: {d}")
    print("  validate_export:", info["validate"])
    pr = info["parity"]
    print(f"  export parity (file reloaded; BF16 embeddings/norms): PPL source model {pr['ppl_trained']:.4f} vs reloaded {pr['ppl_reloaded_export']:.4f} (gap {pr['ppl_gap']:.3%}, gate <= 3%), exported tokenizer gives identical ids/VOCAB.BIN: {pr['token_parity_exported_tokenizer']}; next-token agreement {pr['argmax_agreement_informational']:.2%} (informational) -> {'PASS' if pr['pass'] else 'FAIL'}")
    rc = info["repack_check"]
    print("  repack_ternary.py present here:", rc["ran"], "| " + (f"rc={rc['rc']}, MODEL.SAF tensors {rc.get('MODEL_SAF_tensors')}, scale exported {rc.get('scale_exported')} == MODEL.SAF {rc.get('scale_in_MODEL_SAF')}: {rc.get('scale_roundtrip_ok')}, VOCAB.BIN == engine-port rebuild: {rc.get('vocab_bin_equals_port')}" if rc["ran"] and rc["rc"] == 0 else rc.get("reason", rc.get("stderr", ""))))
    print("  sha256 of every exported file (also in SHA256SUMS):")
    for fn, h in info["sha256"].items():
        print(f"    {h}  {fn}")

if EXPORTS:
    a0 = "step0" if "step0" in EXPORTS else sorted(EXPORTS)[0]
    ED = os.path.join(EXPORT_ROOT, a0)
    n_saf = 7 * NL * 2 + 4 * NL + 1
    prune = "" if FALLBACK_TOKENIZER else "--llama3-prune "
    print(f"""
================ RUN ON THE LINUX SIDE (EXPORT = the path printed by this run, ART = where the repacked artifacts go) ================
EXPORT={ED}      # also: {EXPORT_ROOT}/<arm>/ for each arm
ART=$HOME/e1_artifacts/{a0}
# 0. fetch the outputs of this kernel first:   kaggle kernels output <owner>/e1-evolve-pilot -p ./e1_out
#    export/step0 = the unmodified incumbent through this export path. Repack and measure it FIRST: it is the K12 scale-convention check.
# 1. repack. The scale convention is PINNED (multiply); do not rely on `auto` (config.json keeps quantization_config.quant_method=bitnet, so auto agrees):
python3 aegis-forge/repack_ternary.py $EXPORT $ART {prune}--max-seq 2048 --source-packing unpacked --scale-convention {PINNED_CONVENTION}
#    expect: "weight_scale convention: multiply", "VOCAB.BIN: {ET.vocab_size} tokens, {len(ET.kept_merges)} merges", "MODEL.SAF: {n_saf} tensors"; sha256 of VOCAB.BIN = {vb_sha}
# 2. G1 - the notebook's token ids vs the real Rust encode() (g1_vectors.jsonl is written by Cell 2; tools are in model/kaggle/tools/):
TOKENIZER_RS=aegis-core/src/tokenizer.rs rustc --edition 2024 -O g1_encode_dump.rs -o g1_encode_dump
python3 g1_rust_parity.py {os.path.join(WORK, 'g1_vectors.jsonl')} $ART/VOCAB.BIN ./g1_encode_dump
# 3. identity: the digest must repeat across two runs/hosts (the incumbent's cab11400d737ac4a is NOT expected: the trits differ, see README):
cis_decode $ART/MODEL.SAF $ART/EMBED.BIN $ART/VOCAB.BIN 64 "Once upon a time"
# 4. receipted dev slice through the real gateway (P0, then the T2 diagnostic suite_e1_dev_T2.tsv), then the scorer:
cd alice-aegis/demo/agent-trace/eval
N=24 AEGIS_MODEL=$ART/MODEL.SAF AEGIS_EMBED=$ART/EMBED.BIN AEGIS_VOCAB=$ART/VOCAB.BIN AGENT_TRACE_BIN=../../../aegis-linux/target/release/examples/agent_trace \\
  ./run_suite.sh $EXPORT/suite_e1_dev_P0.tsv /path/out_p0 --table $EXPORT/dev_table.tsv
python3 -I score.py /path/out_p0/summary.tsv
# (FILE-READ, undeclared-tool and abstention items are excluded from the TSV: run_suite.sh only wires --table for LOOKUP items and score.py has no abstain rule yet;
#  distractors are scanned for CALC( only without --table, as in LAB-07 s7.)
# 5. one receipt by hand:   agent_trace gen $ART/MODEL.SAF $ART/EMBED.BIN $ART/VOCAB.BIN 2 24 "<prompt>" --table $EXPORT/dev_table.tsv > r.txt
#                           agent_trace verify $ART/MODEL.SAF $ART/EMBED.BIN $ART/VOCAB.BIN r.txt --table $EXPORT/dev_table.tsv
# Run on box1/box2 via systemd-run with raw logs under state/reports/ (never --allow-busy on penguin). Keep the export private: weights and recipe are PUBLISH? items.
=================================================================================================================""")
print("E1_EXPORT_PASS" if EXPORTS and all(e["parity"]["pass"] for e in EXPORTS.values()) else "E1_EXPORT_FAIL")


--- export arm step0: e1_smoke_out/export/step0
  validate_export: {'projection_weights': 14, 'scales': 14, 'norms': 9, 'tensors': 38}
  export parity (file reloaded; BF16 embeddings/norms): PPL source model 1041889.7343 vs reloaded 1039674.3254 (gap 0.213%, gate <= 3%), exported tokenizer gives identical ids/VOCAB.BIN: True; next-token agreement 91.67% (informational) -> PASS
  repack_ternary.py present here: True | rc=0, MODEL.SAF tensors 37, scale exported 0.645891547203064 == MODEL.SAF 0.645891547203064: True, VOCAB.BIN == engine-port rebuild: True
  sha256 of every exported file (also in SHA256SUMS):
    e6744f6c9b2e203995c8860055d666f7ecf712e1a10a83e4ce5d42141952a061  EXPORT_CONTRACT.json
    f378205dd5cc210265b52821c2f3817f9f684bb9f0597c410aea439a671131a4  config.json
    cee5c127eb5caedb08ac31130b0ac3b458863fe05890d3c5335f30ab3de74a20  dev_table.tsv
    c7eb3f7033301010349326274ef67caf322c9a2a2ef2af8034778850ce0302f3  final_argmax.npy
    94ab14eef7356c1ff27b63753428a4d4cf36d5


--- export arm P: e1_smoke_out/export/P
  validate_export: {'projection_weights': 14, 'scales': 14, 'norms': 9, 'tensors': 38}
  export parity (file reloaded; BF16 embeddings/norms): PPL source model 712843.1082 vs reloaded 703518.1123 (gap 1.308%, gate <= 3%), exported tokenizer gives identical ids/VOCAB.BIN: True; next-token agreement 76.98% (informational) -> PASS
  repack_ternary.py present here: True | rc=0, MODEL.SAF tensors 37, scale exported 0.645891547203064 == MODEL.SAF 0.645891547203064: True, VOCAB.BIN == engine-port rebuild: True
  sha256 of every exported file (also in SHA256SUMS):
    b969b12c60b86b37bcefcac62e5bfbdd08ee92069ef26116050ec9fb7f06a3e9  EXPORT_CONTRACT.json
    f378205dd5cc210265b52821c2f3817f9f684bb9f0597c410aea439a671131a4  config.json
    cee5c127eb5caedb08ac31130b0ac3b458863fe05890d3c5335f30ab3de74a20  dev_table.tsv
    43de929d7864c2bf9d76a49982764bb703c615ffb30757799dca36415b935d02  final_argmax.npy
    3dae6e5180eb0a35b154cf9af1b24a96eecbfa6eb00864d9


--- export arm L: e1_smoke_out/export/L
  validate_export: {'projection_weights': 14, 'scales': 14, 'norms': 9, 'tensors': 38}
  export parity (file reloaded; BF16 embeddings/norms): PPL source model 997297.9117 vs reloaded 1010227.4038 (gap 1.296%, gate <= 3%), exported tokenizer gives identical ids/VOCAB.BIN: True; next-token agreement 79.37% (informational) -> PASS
  repack_ternary.py present here: True | rc=0, MODEL.SAF tensors 37, scale exported 0.645888090133667 == MODEL.SAF 0.645888090133667: True, VOCAB.BIN == engine-port rebuild: True
  sha256 of every exported file (also in SHA256SUMS):
    e405f2baf4b5002442dba38d91d017f4d5c3f38268d3502c7ceff7401a3e295d  EXPORT_CONTRACT.json
    f378205dd5cc210265b52821c2f3817f9f684bb9f0597c410aea439a671131a4  config.json
    cee5c127eb5caedb08ac31130b0ac3b458863fe05890d3c5335f30ab3de74a20  dev_table.tsv
    ab21ef059fe836d053890ae4edb71bf2e0bf8a84d8b68541c8aad06af08ed793  final_argmax.npy
    349b24efe8ec2339f17b98f74a789b556016586615324fa


--- export arm H: e1_smoke_out/export/H
  validate_export: {'projection_weights': 14, 'scales': 14, 'norms': 9, 'tensors': 38}
  export parity (file reloaded; BF16 embeddings/norms): PPL source model 732812.0125 vs reloaded 718782.9745 (gap 1.914%, gate <= 3%), exported tokenizer gives identical ids/VOCAB.BIN: True; next-token agreement 70.63% (informational) -> PASS
  repack_ternary.py present here: True | rc=0, MODEL.SAF tensors 37, scale exported 0.6460427641868591 == MODEL.SAF 0.6460427641868591: True, VOCAB.BIN == engine-port rebuild: True
  sha256 of every exported file (also in SHA256SUMS):
    7cfb8dd442c977cf45e78bc89f4cd5f31c59922c649e48f4de40eb2a9608ce33  EXPORT_CONTRACT.json
    f378205dd5cc210265b52821c2f3817f9f684bb9f0597c410aea439a671131a4  config.json
    cee5c127eb5caedb08ac31130b0ac3b458863fe05890d3c5335f30ab3de74a20  dev_table.tsv
    48ef4269d7253a3b6c87d557cab7d1d19017df0ff2542d39566d8a9d025f9241  final_argmax.npy
    651eff5a749ce45b78f32b849ddf10a2daaf5f3f5ba341

## Cell 5 - results: gates vs measured -> `results.json`

Every row compares one pre-registered **GATE** with the value this notebook measured. Statuses: `PASS` / `FAIL` / `PENDING` (needs the Linux side) / `NOT RUN` (arm skipped or killed before the measurement).
A smoke run prints the same table with `[smoke]` on every status: those rows prove the code path, they say nothing about the model.

In [11]:
QUOTA_AFTER = read_quota()
tag = " [smoke]" if SMOKE else ""
ROWS = []


def row(gid, what, gate, measured, status):
    ROWS.append({"id": gid, "what": what, "gate": gate, "measured": measured, "status": status + tag})


def pf(ok):
    return "PASS" if ok else "FAIL"


done = {a: r for a, r in ARM_RESULTS.items() if r.get("evals")}
base_ppl = math.exp(BASE["mean_nll"])

# G1 / G12 / G5 (cells 0 and 2)
row("G1a", "engine-port vocabulary == incumbent VOCAB.BIN", f"sha256 == {PINNED_VOCAB_SHA256[:12]}...", f"rebuilt {vb_sha[:12]}...", pf(vb_sha == PINNED_VOCAB_SHA256) if not FALLBACK_TOKENIZER else "NOT RUN (toy tokenizer)")
row("G1b", "tokenizer id parity vs the Rust encode() (>= 10 MB)", "100% identical ids", "g1_vectors.jsonl written; comparison runs on the Linux side", "PENDING")
row("G12", "fp16 vs bf16 numerics, 20 prompts", "finite and first-token argmax agreement >= 18/20 [plan GATE]",
    f"finite={G12['finite']}, agree {G12['first_token_argmax_agree']}/20, max |dlogit| {G12['max_abs_logit_diff']:.3f}", pf(G12["pass"]))
row("G5", "CIS activation-quantization points live in the training forward", "all 8 point kinds on the int8 grid", f"{G5['points_checked']} probed inputs", pf(G5["all_on_int8_grid"]))

# gate 1: tokens/s and quota
tps_arms = {a: r.get("train_tokens_per_s_median") for a, r in done.items()}
q_ok = (QUOTA_BEFORE.get("status") not in ("unavailable",)) and (QUOTA_AFTER.get("status") not in ("unavailable",))
row("1", "real tokens/s measured and the quota logged", "both recorded",
    f"base forward {THROUGHPUT['base_forward_tokens_per_s']:.0f} tok/s ({THROUGHPUT['gpu']}); training packed tok/s median {({a: round(v) for a, v in tps_arms.items() if v})}; quota before/after: {QUOTA_BEFORE.get('status')}/{QUOTA_AFTER.get('status')}",
    "PASS" if (tps_arms and q_ok) else ("PARTIAL (tokens/s yes, quota not readable: enter the panel value via E1_QUOTA_BEFORE_H)" if tps_arms else "NOT RUN"))

# gate 2: loss decreases, no NaN
for a, r in ARM_RESULTS.items():
    if r.get("status") == "SKIPPED":
        row(f"2/{a}", f"arm {a}: loss decreases, no NaN", "decrease, finite", r.get("reason", ""), "NOT RUN")
    else:
        ok2 = bool(r.get("loss_decreased")) and r.get("killed_by") != "non-finite loss"
        row(f"2/{a}", f"arm {a}: loss decreases, no NaN", "first-10 mean > last-10 mean, finite",
            (f"{r['loss_first10']:.4f} -> {r['loss_last10']:.4f}; " if r.get("loss_first10") is not None else "") + f"status {r['status']}", pf(ok2))


# gate 3 / 4: best arm
def arm_key(a):
    f = done[a]["final"]
    return (f["ppl_ratio"] <= 1.05, f["p0_unknown_strict"]["rate"], -f["ppl_ratio"])


complete = [a for a in done if done[a]["status"] == "COMPLETE"]
best = max(complete, key=arm_key) if complete else None
if best:
    f, b0 = done[best]["final"], done[best]["baseline_step0"]
    p0 = f["p0_unknown_strict"]
    dist, dist0 = f["P0"]["distractor"], b0["P0"]["distractor"]
    g3 = (p0["rate"] >= 0.733) and (dist["rate"] + 1e-9 >= dist0["rate"])
    row("3", f"best arm ({best}): P0 unknown_fact exact-key rate and distractor no-tool precision", ">= 73.3% and distractor >= the baseline value of the same harness",
        f"unknown_fact {p0['k']}/{p0['n']} = {p0['rate']:.1%} (Wilson {p0['wilson'][0]:.0%}-{p0['wilson'][1]:.0%}); step-0 {b0['p0_unknown_strict']['rate']:.1%}; distractor {dist['k']}/{dist['n']} vs step-0 {dist0['k']}/{dist0['n']}", pf(g3))
    row("4", f"arm {best}: held-out PPL ratio vs the incumbent", "<= 1.05 (flag above 1.02)", f"{f['ppl_ratio']:.4f} (incumbent PPL {base_ppl:.3f} on {HELD_SRC.split(' (')[0]})",
        pf(f["ppl_ratio"] <= 1.05) + (" [flag > 1.02]" if f["ppl_ratio"] > 1.02 else ""))
else:
    row("3", "best arm: P0 unknown_fact + distractor", ">= 73.3%", "no arm completed", "NOT RUN")
    row("4", "best arm: PPL ratio", "<= 1.05", "no arm completed", "NOT RUN")
for a in done:   # step-0 parity gate of every arm (the incumbent through this notebook's QAT forward vs the stock forward)
    b0 = done[a]["baseline_step0"]
    row(f"0/{a}", f"arm {a}: step-0 parity vs the stock reference ({BASE['kind']})", "argmax disagreement <= 3%", f"{b0['parity_disagreement']:.2%}; PPL ratio {b0['ppl_ratio']:.4f}", pf(b0["parity_disagreement"] <= 0.03))

# gate 5: export parity
for a, e in EXPORTS.items():
    p = e["parity"]
    row(f"5/{a}", ("step-0 (unmodified incumbent)" if a == "step0" else f"arm {a}") + ": export parity (file reloaded, BF16 embeddings/norms)", "PPL gap <= 3% and token parity",
        f"PPL gap {p['ppl_gap']:.3%}; exported tokenizer reproduces ids: {p['token_parity_exported_tokenizer']}; next-token agreement {p['argmax_agreement_informational']:.2%} (informational)", pf(p["pass"]))
if not EXPORTS:
    row("5", "step-0 export parity", "PPL gap <= 3%, token parity", "no export", "NOT RUN")
row("5b", "engine parity (cis_decode, receipts, HF-vs-engine PPL gap)", "<= 3% PPL gap, digest repeats", "Linux side (commands printed by Cell 4)", "PENDING")

# recorded, not gated (6)
for a, r in done.items():
    t = r["final"].get("ternary") or {}
    row(f"6/{a}", f"arm {a}: trit-flip fraction and p(0) census (recorded, not gated)", "-", f"flips {t.get('flip_fraction', 0):.2e} (max tensor {t.get('flip_fraction_max_tensor', 0):.2e}); p(0) {t.get('p0_census')}", "RECORDED")
if "P" in done and best and best != "P":
    gP = done["P"]["final"]["p0_unknown_strict"]["rate"] - done["P"]["baseline_step0"]["p0_unknown_strict"]["rate"]
    gB = done[best]["final"]["p0_unknown_strict"]["rate"] - done[best]["baseline_step0"]["p0_unknown_strict"]["rate"]
    row("6/capture", "does periphery-only (arm P) capture most of the gain?", "- (recorded)", f"unknown_fact gain P {gP:+.1%} vs best arm {best} {gB:+.1%}" + (f" -> captures {gP / gB:.0%}" if gB > 0 else ""), "RECORDED")

# ---- print
w = max(len(r["id"]) for r in ROWS)
print(("SMOKE RUN: every value below proves a code path and is NOT a measurement\n" if SMOKE else "") + "=" * 150)
for r in ROWS:
    print(f"[{r['id']:>{w}}] {r['status']:<34} | {r['what']}\n{'':>{w + 3}} gate: {r['gate']}\n{'':>{w + 3}} measured: {r['measured']}")
print("=" * 150)


def compact(r):
    if not r.get("evals"):
        return {k: r.get(k) for k in ("status", "reason", "plan")}
    f, b = r["final"], r["baseline_step0"]
    return {"status": r["status"], "killed_by": r.get("killed_by"), "lr": r["lr"], "steps_done": r["steps_done"], "total_steps": r["total_steps"], "trainable_params": r["trainable_params"],
            "optimizer": r["optimizer"], "loss_first10": r["loss_first10"], "loss_last10": r["loss_last10"], "loss_decreased": r["loss_decreased"],
            "train_tokens_per_s_median": r["train_tokens_per_s_median"], "baseline_step0": {k: b[k] for k in ("heldout_ppl", "ppl_ratio", "parity_disagreement", "p0_unknown_strict", "P0", "T2") if k in b},
            "final": {k: f[k] for k in ("step", "heldout_ppl", "ppl_ratio", "parity_disagreement", "p0_unknown_strict", "P0", "T2", "ternary") if k in f},
            "eval_curve": [{"step": e["step"], "ppl_ratio": e["ppl_ratio"], "p0_unknown": e["p0_unknown_strict"]["rate"], "parity": e["parity_disagreement"]} for e in r["evals"]]}


RESULTS = {
    "notebook": "e1-evolve-pilot", "smoke": SMOKE, "claims": "none: smoke plumbing only" if SMOKE else "measurements of this run only; nothing here is a product number",
    "env": ENV, "quota": {"before": QUOTA_BEFORE, "after": QUOTA_AFTER}, "throughput": THROUGHPUT, "g12": G12, "g5": G5, "qat_vs_hf_layer": QEQ, "trits_vs_packed_incumbent": TRITS_VS_PACKED,
    "tokenizer": {"kind": "engine-port", "vocab_bin_rebuilt_sha256": vb_sha, "pinned_incumbent_vocab_bin_sha256": PINNED_VOCAB_SHA256, "pruned_vocab": ET.vocab_size, "merges": len(ET.kept_merges),
                  "toy_fallback": FALLBACK_TOKENIZER, "skew_cases": skew_report},
    "dataset": {"manifest_sha256": sha256_file(os.path.join(DATA_DIR, "manifest.json")), "files": man["files"], "n_train_episodes": man.get("n_train_episodes"), "real_tokens": man.get("real_tokens"),
                "generators": man.get("generators"), "dev_buckets": man.get("dev_buckets"), "seq_len": SEQ},
    "heldout": {"source": HELD_SRC, "sha256": HELD_SHA, "windows": int(HELD.shape[0]), "window_tokens": int(HELD.shape[1])},
    "base_reference": {"kind": BASE["kind"], "heldout_ppl": base_ppl},
    "cfg": {k: v for k, v in CFG.items()}, "arms": {a: compact(r) for a, r in ARM_RESULTS.items()}, "best_arm": best,
    "exports": {a: {"dir": e["dir"], "validate": e["validate"], "parity": e["parity"], "suites": e["suites"], "repack_check": e["repack_check"], "sha256": e["sha256"]} for a, e in EXPORTS.items()},
    "gates": ROWS,
    "honesty": ["every number is a measurement from this run (or labelled)", "nothing is a product number", "Rule A: no timing claims outside the GPU name / tokens-per-second logging lines",
                "receipts for the eval items are produced by the Linux gateway, not by this notebook; the scorer here is a proxy"],
}
json.dump(RESULTS, open(os.path.join(WORK, "results.json"), "w"), indent=1, default=str)
print("results.json sha256:", sha256_file(os.path.join(WORK, "results.json")))
loss_ok = any(r.get("loss_decreased") and r.get("status") in ("COMPLETE", "KILLED") for r in ARM_RESULTS.values())
print("E1_TRAIN_PASS" if loss_ok else "E1_TRAIN_FAIL")

SMOKE RUN: every value below proves a code path and is NOT a measurement
[    G1a] PASS [smoke]                       | engine-port vocabulary == incumbent VOCAB.BIN
           gate: sha256 == 5bde1b0355ef...
           measured: rebuilt 5bde1b0355ef...
[    G1b] PENDING [smoke]                    | tokenizer id parity vs the Rust encode() (>= 10 MB)
           gate: 100% identical ids
           measured: g1_vectors.jsonl written; comparison runs on the Linux side
[    G12] PASS [smoke]                       | fp16 vs bf16 numerics, 20 prompts
           gate: finite and first-token argmax agreement >= 18/20 [plan GATE]
           measured: finite=True, agree 19/20, max |dlogit| 4.973
[     G5] PASS [smoke]                       | CIS activation-quantization points live in the training forward
           gate: all 8 point kinds on the int8 grid
           measured: 15 probed inputs
[      1] PASS [smoke]                       | real tokens/s measured and the quota logged
           ga

## Honesty rules for this notebook (read before quoting anything from it)

1. **Every number here is a measurement made by this notebook in this session**, or it is explicitly labelled as quoted from the plan (`[arith]`, `[est]`, LAB-07). Nothing printed is a product number and nothing here qualifies the model for shipping:
   the dev slice is about 40-60 items per bucket, so every interval is wide, and gate 3 is a *movement* test against the same harness's step-0 baseline, not a ship test.
2. **Rule A:** no timing claim is made anywhere except the GPU name / tokens-per-second logging lines (`E1_THROUGHPUT`, the per-log-step `packed_tokens_per_s`). Those are Kaggle-hardware training throughputs, not engine speeds, and not comparable with any decode figure.
3. **Rule B:** every number in `results.json` traces to a log line, the frozen dataset manifest (SHA-256), the held-out slice hash, or an exported file hash. A number without that trail is not to be quoted.
4. **The receipts for the eval items are produced by the Linux gateway (`agent_trace gen` / `verify`), not by this notebook.** The scorer in Cell 3 is a proxy of `score.py` over 24 greedy tokens; it can rank checkpoints but it is not the instrument of record. `score_ext.py` (abstain list, legality) does not exist yet.
5. **Reference and reading caveats.** The perplexity/parity reference is the stock Hugging Face online-BitLinear forward of the bf16 master, not the engine; the engine comparison (HF-vs-engine PPL gap, digest repeatability) is Linux-side. The P0 reading ("one-line declaration, zero example calls") is the plan author's (K18) and awaits Justin's confirmation.
   Held-out text is the program's PPL anchor if WikiText-2 was reachable, otherwise synthetic prose that is **not** comparable with it (stated in `results.json`).
6. **Provenance and publication.** The base's lineage (named SFT/DPO sources) and the Llama-3 tokenizer notice are unresolved (file 03 A1-A4); the weights, the recipe and the factory stay private until a `PUBLISH?` item is answered. Keep this kernel private.
7. **A smoke run is not a measurement.** `E1_SMOKE=1` runs a tiny random network on CPU; its gates exist to show the code runs, and every status it prints carries `[smoke]`.